In [1]:

# h5ad 구조 확인

import scanpy as sc
adata = sc.read_h5ad('/project_antwerp/hbae/data/Open_ST/GSM7990099_primary_HNSCC.h5ad')
print('Shape:', adata.shape)
print('obs columns:', list(adata.obs.columns))
print('obsm keys:', list(adata.obsm.keys()))
print('uns keys:', list(adata.uns.keys()))
print('X dtype:', adata.X.dtype)
# spatial coordinates 확인
if 'spatial' in adata.obsm:
    print('spatial coords range:', adata.obsm['spatial'].min(axis=0), adata.obsm['spatial'].max(axis=0))
elif 'x' in adata.obs.columns:
    print('x range:', adata.obs['x'].min(), adata.obs['x'].max())
    print('y range:', adata.obs['y'].min(), adata.obs['y'].max())


Shape: (36642, 27095)
obs columns: ['cell_ID_mask', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_reads', 'reads_per_counts', 'n_joined', 'exact_entropy', 'theoretical_entropy', 'exact_compression', 'theoretical_compression', 'n_counts', 'annotation']
obsm keys: ['spatial']
uns keys: ['log1p', 'spatial']
X dtype: float64
spatial coords range: [1350.70097087 3195.98476771] [ 8830.44826224 15042.0403481 ]


In [2]:

import scanpy as sc
import numpy as np
adata = sc.read_h5ad('/project_antwerp/hbae/data/Open_ST/GSM7990099_primary_HNSCC.h5ad')

# spatial 좌표 vs 이미지 크기 확인
import tifffile
img = tifffile.imread('/project_antwerp/hbae/data/Open_ST/GSM7990099_primary_HNSCC.tif')
print('Image shape:', img.shape)
print('Image dtype:', img.dtype)

coords = adata.obsm['spatial']
print('Coords shape:', coords.shape)
print('Coords min:', coords.min(axis=0))
print('Coords max:', coords.max(axis=0))
print('First 5 coords:', coords[:5])

# uns spatial 확인
if 'spatial' in adata.uns:
    print('uns spatial keys:', list(adata.uns['spatial'].keys()))

# layers 확인
print('layers:', list(adata.layers.keys()))

Image shape: (10501, 16686, 3)
Image dtype: uint8
Coords shape: (36642, 2)
Coords min: [1350.70097087 3195.98476771]
Coords max: [ 8830.44826224 15042.0403481 ]
First 5 coords: [[ 1350.70097087 12331.23106796]
 [ 1369.19140083 10921.86199723]
 [ 1371.02102102 10983.58858859]
 [ 1377.27687983 12230.72171469]
 [ 1394.35993899 12344.00610066]]
uns spatial keys: ['mask', 'tissue']
layers: ['raw']


In [3]:

import scanpy as sc
import numpy as np
import tifffile

adata = sc.read_h5ad('/project_antwerp/hbae/data/Open_ST/GSM7990099_primary_HNSCC.h5ad')
img = tifffile.imread('/project_antwerp/hbae/data/Open_ST/GSM7990099_primary_HNSCC.tif')
mask = np.load('/project_antwerp/hbae/data/Open_ST/GSM7990099_primary_HNSCC/mask.npy')

print('Image shape (H, W, C):', img.shape)
print('Mask shape:', mask.shape)
print('Mask unique values:', np.unique(mask)[:10])
print('Mask dtype:', mask.dtype)

# coords: col0=x, col1=y ?
coords = adata.obsm['spatial']
x, y = coords[:, 0], coords[:, 1]
print(f'x range: {x.min():.1f} - {x.max():.1f}')
print(f'y range: {y.min():.1f} - {y.max():.1f}')
print(f'Image H={img.shape[0]}, W={img.shape[1]}')
# x ~ W방향, y ~ H방향 맞는지 확인
print(f'x/W = {x.max()/img.shape[1]:.3f}, y/H = {y.max()/img.shape[0]:.3f}')


Image shape (H, W, C): (10501, 16686, 3)
Mask shape: (82, 130)
Mask unique values: [0 1]
Mask dtype: uint8
x range: 1350.7 - 8830.4
y range: 3196.0 - 15042.0
Image H=10501, W=16686
x/W = 0.529, y/H = 1.432


In [4]:

import scanpy as sc
import numpy as np
import tifffile
from scipy.ndimage import zoom

adata = sc.read_h5ad('/project_antwerp/hbae/data/Open_ST/GSM7990099_primary_HNSCC.h5ad')
img = tifffile.imread('/project_antwerp/hbae/data/Open_ST/GSM7990099_primary_HNSCC.tif')
mask = np.load('/project_antwerp/hbae/data/Open_ST/GSM7990099_primary_HNSCC/mask.npy')

# mask를 이미지 크기로 upsample
scale_h = img.shape[0] / mask.shape[0]
scale_w = img.shape[1] / mask.shape[1]
print(f'Scale factors: H={scale_h:.2f}, W={scale_w:.2f}')

mask_full = zoom(mask, (scale_h, scale_w), order=0)
print(f'Upsampled mask shape: {mask_full.shape}')

# coords[:,0]=row, coords[:,1]=col 가정하고 검증
# 모든 cell이 mask=1 영역 안에 있어야 함
coords = adata.obsm['spatial']
rows = coords[:, 0].astype(int)
cols = coords[:, 1].astype(int)

# clip to image bounds
rows_c = np.clip(rows, 0, img.shape[0]-1)
cols_c = np.clip(cols, 0, img.shape[1]-1)
in_mask = mask_full[rows_c, cols_c]
print(f'Cells in mask (coords[:,0]=row): {in_mask.mean():.3f}')

# 반대로도 테스트
rows2 = coords[:, 1].astype(int)
cols2 = coords[:, 0].astype(int)
rows2_c = np.clip(rows2, 0, img.shape[0]-1)
cols2_c = np.clip(cols2, 0, img.shape[1]-1)
in_mask2 = mask_full[rows2_c, cols2_c]
print(f'Cells in mask (coords[:,1]=row): {in_mask2.mean():.3f}')


Scale factors: H=128.06, W=128.35
Upsampled mask shape: (10501, 16686)
Cells in mask (coords[:,0]=row): 0.984
Cells in mask (coords[:,1]=row): 0.505


In [1]:

import numpy as np
from PIL import Image

npy = np.load('/project_antwerp/hbae/data/Open_ST/GSM7990099_primary_HNSCC/mask.npy')
png = np.array(Image.open('/project_antwerp/hbae/data/Open_ST/GSM7990099_primary_HNSCC/mask.png'))

print('npy shape:', npy.shape, '| dtype:', npy.dtype, '| unique:', np.unique(npy))
print('png shape:', png.shape, '| dtype:', png.dtype, '| unique:', np.unique(png))


npy shape: (82, 130) | dtype: uint8 | unique: [0 1]
png shape: (82, 130) | dtype: uint8 | unique: [  0 255]


In [2]:

import numpy as np
import tifffile
from scipy.ndimage import zoom
from PIL import Image

img  = tifffile.imread('/project_antwerp/hbae/data/Open_ST/GSM7990099_primary_HNSCC.tif')
mask = np.load('/project_antwerp/hbae/data/Open_ST/GSM7990099_primary_HNSCC/mask.npy')

H, W = img.shape[:2]
mask_full = zoom(mask, (H/mask.shape[0], W/mask.shape[1]), order=0).astype(np.uint8)

# masked image 작은 썸네일로 저장해서 확인
mask_3ch   = np.stack([mask_full]*3, axis=-1)
img_masked = np.where(mask_3ch == 1, img, 255).astype(np.uint8)

# 1/20 축소해서 확인용 저장
thumb = Image.fromarray(img_masked[::20, ::20])
thumb.save('/project_antwerp/hbae/data/Open_ST/masked_thumbnail.png')
print('Saved thumbnail:', thumb.size)


Saved thumbnail: (835, 526)


In [3]:

import scanpy as sc
import numpy as np

adata = sc.read_h5ad('/project_antwerp/hbae/data/Open_ST/GSM7990099_primary_HNSCC.h5ad')

# raw layer 값 범위 확인
import scipy.sparse as sp
raw = adata.layers['raw']
if sp.issparse(raw): raw = raw.toarray()

print('layers[raw] min/max/mean:', raw.min(), raw.max(), raw.mean())
print('layers[raw] 첫 cell 값 샘플:', raw[0, raw[0]>0][:10])

# X 원본 값도 확인
X = adata.X
if sp.issparse(X): X = X.toarray()
print('X min/max/mean:', X.min(), X.max(), X.mean())
print('X 첫 cell 값 샘플:', X[0, X[0]>0][:10])


layers[raw] min/max/mean: 0.0 884.0 0.09139927973891691
layers[raw] 첫 cell 값 샘플: [1. 1. 1. 1. 6. 3. 1. 1. 1. 1.]
X min/max/mean: 0.0 6.656239652258941 0.038009903544182606
X 첫 cell 값 샘플: [1.65247542 1.65247542 1.65247542 1.65247542 3.27030295 2.61444666
 1.65247542 1.65247542 1.65247542 1.65247542]


In [4]:

import numpy as np
import h5py

# Open-ST gene list
with h5py.File('/project_antwerp/hbae/data/Open_ST/openst_patches.h5', 'r') as f:
    openst_genes = [g.decode() if isinstance(g, bytes) else g 
                    for g in f.attrs['gene_names']]

# HVG 2000
hvg_genes = open('/project_antwerp/hbae/data/0317_hvg_2000_list.txt').read().strip().split('\n')

# 겹치는 gene
shared = [g for g in hvg_genes if g in set(openst_genes)]
missing = [g for g in hvg_genes if g not in set(openst_genes)]

print(f'HVG 2000 genes    : {len(hvg_genes)}')
print(f'Open-ST genes     : {len(openst_genes)}')
print(f'Shared            : {len(shared)}')
print(f'Missing from Open-ST: {len(missing)}')
print(f'Missing examples  : {missing[:10]}')


HVG 2000 genes    : 2000
Open-ST genes     : 1946
Shared            : 1946
Missing from Open-ST: 54
Missing examples  : ['AC073111.4', 'ACTC1', 'ADORA2A', 'AGR2', 'ATP5MPL', 'C10orf99', 'CD99', 'CDC42EP2', 'CEACAM7', 'COQ8A']


In [5]:

import numpy as np
import h5py

with h5py.File('/project_antwerp/hbae/data/Open_ST/openst_patches.h5', 'r') as f:
    val_exprs  = f['expression'][:]
    openst_genes = [g.decode() if isinstance(g, bytes) else g 
                    for g in f.attrs['gene_names']]

# Open-ST 기준 top-300
mean_expr  = val_exprs.mean(axis=0)
top300_idx = np.argsort(mean_expr)[::-1][:300]
openst_top300 = set([openst_genes[i] for i in top300_idx])

# Visium val set top-300 가져오기 (fold_01 기준)
val_exprs_visium = np.load('/project_antwerp/hbae/Loki_output/0317_epoch10_finetune_embedding_new/fold_01/val_exprs.npy')
hvg_genes = open('/project_antwerp/hbae/data/0317_hvg_2000_list.txt').read().strip().split('\n')
mean_visium = val_exprs_visium.mean(axis=0)
top300_visium_idx = np.argsort(mean_visium)[::-1][:300]
visium_top300 = set([hvg_genes[i] for i in top300_visium_idx])

overlap = openst_top300 & visium_top300
print(f'Open-ST top-300   : {len(openst_top300)}')
print(f'Visium top-300    : {len(visium_top300)}')
print(f'Overlap           : {len(overlap)}')
print(f'Only in Open-ST   : {len(openst_top300 - visium_top300)}')
print(f'Only in Visium    : {len(visium_top300 - openst_top300)}')


Open-ST top-300   : 300
Visium top-300    : 300
Overlap           : 194
Only in Open-ST   : 106
Only in Visium    : 106


In [6]:

import numpy as np
import h5py
import torch
import torch.nn.functional as F

# 1. patch 샘플 확인 - 실제 이미지가 제대로 들어갔는지
with h5py.File('/project_antwerp/hbae/data/Open_ST/openst_patches.h5', 'r') as f:
    patches = f['patches'][:5]   # 첫 5개
    expr    = f['expression'][:5]
    print('Patch shape:', patches.shape)
    print('Patch min/max/mean:', patches.min(), patches.max(), patches.mean())
    print('Expression min/max/mean:', expr.min(), expr.max(), expr.mean())
    print('Expression first cell nonzero count:', (expr[0] > 0).sum())

# 2. embedding 분포 확인
val_emb   = np.load('/project_antwerp/hbae/Loki_output/openst_validation/fold_01/openst_img_embs.npy')
train_emb = np.load('/project_antwerp/hbae/Loki_output/0317_epoch10_finetune_embedding_new/fold_01/train_img_embs.npy')

val_emb_t   = torch.tensor(val_emb)
train_emb_t = torch.tensor(train_emb)
val_norm   = F.normalize(val_emb_t, dim=-1)
train_norm = F.normalize(train_emb_t, dim=-1)

# cosine similarity 분포
sim = val_norm[:100] @ train_norm.T   # 100개 샘플
print('\nCosine similarity (val vs train):')
print('  mean:', sim.mean().item())
print('  max:', sim.max().item())
print('  min:', sim.min().item())

# Visium val embedding과 비교
import os
val_visium = np.load('/project_antwerp/hbae/Loki_output/0317_epoch10_finetune_embedding_new/fold_01/val_img_embs.npy')
val_vis_t  = torch.tensor(val_visium)
val_vis_norm = F.normalize(val_vis_t, dim=-1)

sim_vis = val_vis_norm[:100] @ train_norm.T
print('\nCosine similarity (Visium val vs train):')
print('  mean:', sim_vis.mean().item())
print('  max:', sim_vis.max().item())
print('  min:', sim_vis.min().item())


Patch shape: (5, 224, 224, 3)
Patch min/max/mean: 1 255 168.81039275085033
Expression min/max/mean: 0.0 6.495503 0.1964165
Expression first cell nonzero count: 75

Cosine similarity (val vs train):
  mean: 0.2192077785730362
  max: 0.6346694827079773
  min: -0.07019509375095367

Cosine similarity (Visium val vs train):
  mean: 0.24013032019138336
  max: 0.9460521936416626
  min: -0.17589178681373596


In [7]:

import numpy as np
import h5py

with h5py.File('/project_antwerp/hbae/data/Open_ST/openst_patches.h5', 'r') as f:
    val_exprs  = f['expression'][:]
    openst_genes = [g.decode() if isinstance(g, bytes) else g 
                    for g in f.attrs['gene_names']]

# top-300 gene들의 sparsity 확인
mean_expr  = val_exprs.mean(axis=0)
top300_idx = np.argsort(mean_expr)[::-1][:300].copy()

top300_expr = val_exprs[:, top300_idx]
top300_genes = [openst_genes[i] for i in top300_idx]

# 각 gene의 nonzero cell 비율
nonzero_ratio = (top300_expr > 0).mean(axis=0)
print('Top-300 gene nonzero ratio:')
print(f'  mean: {nonzero_ratio.mean():.3f}')
print(f'  min:  {nonzero_ratio.min():.3f}')
print(f'  <10% nonzero: {(nonzero_ratio < 0.1).sum()} genes')
print(f'  <5%  nonzero: {(nonzero_ratio < 0.05).sum()} genes')

# Visium val과 sparsity 비교
val_visium = np.load('/project_antwerp/hbae/Loki_output/0317_epoch10_finetune_embedding_new/fold_01/val_exprs.npy')
hvg_genes = open('/project_antwerp/hbae/data/0317_hvg_2000_list.txt').read().strip().split('\n')
mean_vis = val_visium.mean(axis=0)
top300_vis_idx = np.argsort(mean_vis)[::-1][:300].copy()
top300_vis_expr = val_visium[:, top300_vis_idx]
nonzero_vis = (top300_vis_expr > 0).mean(axis=0)
print('\nVisium val top-300 gene nonzero ratio:')
print(f'  mean: {nonzero_vis.mean():.3f}')
print(f'  min:  {nonzero_vis.min():.3f}')
print(f'  <10% nonzero: {(nonzero_vis < 0.1).sum()} genes')

# cell당 nonzero gene 수
print(f'\nOpen-ST cells nonzero genes (top-300): mean={( top300_expr > 0).sum(axis=1).mean():.1f}')
print(f'Visium spots nonzero genes (top-300): mean={(top300_vis_expr > 0).sum(axis=1).mean():.1f}')


Top-300 gene nonzero ratio:
  mean: 0.404
  min:  0.151
  <10% nonzero: 0 genes
  <5%  nonzero: 0 genes

Visium val top-300 gene nonzero ratio:
  mean: 0.738
  min:  0.398
  <10% nonzero: 0 genes

Open-ST cells nonzero genes (top-300): mean=121.3
Visium spots nonzero genes (top-300): mean=221.5


In [8]:

import numpy as np
import h5py
import tifffile
from scipy.ndimage import zoom
from PIL import Image, ImageDraw
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# 데이터 로드
with h5py.File('/project_antwerp/hbae/data/Open_ST/openst_patches.h5', 'r') as f:
    patches    = f['patches'][:9]       # 첫 9개
    rows_valid = f['coords_row'][:9]
    cols_valid = f['coords_col'][:9]

# 원본 이미지에서 동일 위치 확인용 썸네일
img  = tifffile.imread('/project_antwerp/hbae/data/Open_ST/GSM7990099_primary_HNSCC.tif')
mask = np.load('/project_antwerp/hbae/data/Open_ST/GSM7990099_primary_HNSCC/mask.npy')
H, W = img.shape[:2]
mask_full = zoom(mask, (H/mask.shape[0], W/mask.shape[1]), order=0).astype(np.uint8)
mask_3ch  = np.stack([mask_full]*3, axis=-1)
img_masked = np.where(mask_3ch==1, img, 255).astype(np.uint8)

# 패치 시각화
fig, axes = plt.subplots(3, 3, figsize=(9, 9))
for i, ax in enumerate(axes.flat):
    ax.imshow(patches[i])
    ax.set_title(f'row={rows_valid[i]}, col={cols_valid[i]}', fontsize=7)
    ax.axis('off')
plt.suptitle('Open-ST patches (224x224)', fontsize=12)
plt.tight_layout()
plt.savefig('/project_antwerp/hbae/data/Open_ST/patch_check_grid.png', dpi=100)
print('Saved patch grid')

# 전체 이미지에서 cell 위치 확인 (썸네일에 점 찍기)
thumb = Image.fromarray(img_masked[::20, ::20])
draw  = ImageDraw.Draw(thumb)
with h5py.File('/project_antwerp/hbae/data/Open_ST/openst_patches.h5', 'r') as f:
    all_rows = f['coords_row'][:]
    all_cols = f['coords_col'][:]

# 썸네일 스케일로 변환 (1/20)
for r, c in zip(all_rows[::50], all_cols[::50]):   # 50개마다 1개
    draw.ellipse([c//20-2, r//20-2, c//20+2, r//20+2], fill='red')
thumb.save('/project_antwerp/hbae/data/Open_ST/cell_locations_thumb.png')
print('Saved cell location thumbnail')
print(f'Total cells plotted: {len(all_rows[::50])}')


Saved patch grid
Saved cell location thumbnail
Total cells plotted: 721


In [9]:

import numpy as np
import h5py
import tifffile
from scipy.ndimage import zoom
from PIL import Image, ImageDraw
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

img   = tifffile.imread('/project_antwerp/hbae/data/Open_ST/GSM7990099_primary_HNSCC.tif')
mask  = np.load('/project_antwerp/hbae/data/Open_ST/GSM7990099_primary_HNSCC/mask.npy')
H, W  = img.shape[:2]
mask_full = zoom(mask, (H/mask.shape[0], W/mask.shape[1]), order=0).astype(np.uint8)
mask_3ch  = np.stack([mask_full]*3, axis=-1)
img_masked = np.where(mask_3ch==1, img, 255).astype(np.uint8)

with h5py.File('/project_antwerp/hbae/data/Open_ST/openst_patches.h5', 'r') as f:
    all_rows = f['coords_row'][:]
    all_cols = f['coords_col'][:]
    all_patches = f['patches']

    # tissue 안쪽 cell 찾기 - 경계에서 멀리 있는 것
    # patch 안에 흰색 픽셀이 거의 없는 cell 선택
    interior_idx = []
    for i in range(len(all_rows)):
        p = f['patches'][i]
        white_ratio = (p > 240).mean()
        if white_ratio < 0.01:   # 흰색 1% 미만
            interior_idx.append(i)
        if len(interior_idx) >= 9:
            break

    print(f'Interior cells found: {len(interior_idx)}')
    patches_interior = f['patches'][interior_idx]
    rows_interior = all_rows[interior_idx]
    cols_interior = all_cols[interior_idx]

# 시각화
fig, axes = plt.subplots(3, 3, figsize=(9, 9))
for i, ax in enumerate(axes.flat):
    # patch에 중심점 표시
    p = patches_interior[i].copy()
    img_pil = Image.fromarray(p)
    draw = ImageDraw.Draw(img_pil)
    cx, cy = 112, 112   # 중심
    draw.ellipse([cx-5, cy-5, cx+5, cy+5], outline='red', width=2)
    draw.line([cx-10, cy, cx+10, cy], fill='red', width=1)
    draw.line([cx, cy-10, cx, cy+10], fill='red', width=1)
    ax.imshow(np.array(img_pil))
    ax.set_title(f'row={rows_interior[i]}, col={cols_interior[i]}', fontsize=7)
    ax.axis('off')

plt.suptitle('Interior patches with center marker', fontsize=12)
plt.tight_layout()
plt.savefig('/project_antwerp/hbae/data/Open_ST/patch_interior_check.png', dpi=100)
print('Saved interior patch check')


Interior cells found: 9
Saved interior patch check


In [10]:

import tifffile
img = tifffile.imread('/project_antwerp/hbae/data/Open_ST/GSM7990099_primary_HNSCC.tif')
print('dtype:', img.dtype)
print('shape:', img.shape)

# tifffile로 메타데이터 확인
with tifffile.TiffFile('/project_antwerp/hbae/data/Open_ST/GSM7990099_primary_HNSCC.tif') as tif:
    page = tif.pages[0]
    print('is_tiled:', page.is_tiled)
    print('tile_width:', getattr(page, 'tilewidth', None))
    print('tile_length:', getattr(page, 'tilelength', None))
    print('compression:', page.compression)
    print('photometric:', page.photometric)

# 실제 픽셀값 확인 - 격자 경계 부근
print()
print('pixel values around row=1731, col=11593:')
r, c = 1731, 11593
print(img[r-2:r+3, c-2:c+3, 0])


dtype: uint8
shape: (10501, 16686, 3)
is_tiled: True
tile_width: 256
tile_length: 256
compression: 8
photometric: 2

pixel values around row=1731, col=11593:
[[ 59  47  53  81  90]
 [ 52  44  52  84 100]
 [ 49  44  49  71  92]
 [ 47  49  50  55  73]
 [ 46  47  44  46  64]]


In [11]:

import tifffile
import numpy as np

img = tifffile.imread('/project_antwerp/hbae/data/Open_ST/GSM7990099_primary_HNSCC.tif')

# tile 경계 (256의 배수) 근처 픽셀 연속성 확인
# row=256 경계
print('Across tile boundary at row=256:')
print(img[254:259, 1000:1005, 0])

# col=256 경계  
print('Across tile boundary at col=256:')
print(img[1000:1005, 254:259, 0])

# 전체 이미지 통계
print(f'Global mean: {img.mean():.2f}')
print(f'Global std:  {img.std():.2f}')
print('No NaN:', not np.isnan(img.astype(float)).any())


Across tile boundary at row=256:
[[ 51  51  52  57  61]
 [ 51  50  53  59  65]
 [101 100 101 104 107]
 [102 101 103 107 110]
 [103 103 105 109 111]]
Across tile boundary at col=256:
[[112 113 131 133 134]
 [114 114 133 135 136]
 [113 113 133 136 137]
 [110 109 132 135 138]
 [108 106 132 134 137]]
Global mean: 91.56
Global std:  26.97
No NaN: True


In [12]:

import tifffile
import numpy as np

# 여러 tile 경계 확인
img = tifffile.imread('/project_antwerp/hbae/data/Open_ST/GSM7990099_primary_HNSCC.tif')

print('=== Tile boundary discontinuity check ===')
for boundary in [256, 512, 768, 1024]:
    diff_row = abs(img[boundary, 1000, 0].astype(int) - img[boundary-1, 1000, 0].astype(int))
    diff_col = abs(img[1000, boundary, 0].astype(int) - img[1000, boundary-1, 0].astype(int))
    print(f'Row boundary {boundary}: pixel jump = {diff_row}')
    print(f'Col boundary {boundary}: pixel jump = {diff_col}')

# 랜덤 non-boundary 위치와 비교
diffs_normal = []
for r in range(300, 800, 10):
    if r % 256 != 0 and r % 256 != 255:
        diff = abs(img[r, 1000, 0].astype(int) - img[r-1, 1000, 0].astype(int))
        diffs_normal.append(diff)
print(f'Normal row diffs mean: {np.mean(diffs_normal):.2f}')

# Open-ST 공식 데이터 처리 방식 확인
with tifffile.TiffFile('/project_antwerp/hbae/data/Open_ST/GSM7990099_primary_HNSCC.tif') as tif:
    print(f'Number of pages: {len(tif.pages)}')
    print(f'Number of series: {len(tif.series)}')
    for i, s in enumerate(tif.series):
        print(f'  Series {i}: shape={s.shape}, dtype={s.dtype}, name={s.name}')


=== Tile boundary discontinuity check ===
Row boundary 256: pixel jump = 50
Col boundary 256: pixel jump = 18
Row boundary 512: pixel jump = 46
Col boundary 512: pixel jump = 30
Row boundary 768: pixel jump = 70
Col boundary 768: pixel jump = 15
Row boundary 1024: pixel jump = 27
Col boundary 1024: pixel jump = 40
Normal row diffs mean: 5.90
Number of pages: 8
Number of series: 1
  Series 0: shape=(10501, 16686, 3), dtype=uint8, name=


In [15]:
import tifffile
import numpy as np

with tifffile.TiffFile('/project_antwerp/hbae/data/Open_ST/GSM7990099_primary_HNSCC.tif') as tif:
    for i, page in enumerate(tif.pages):
        print(
            f"Page {i}: shape={page.shape}, dtype={page.dtype}, "
            f"is_tiled={page.is_tiled}, "
            f"tile=({getattr(page, 'tilelength', None)}, {getattr(page, 'tilewidth', None)}), "
            f"compression={page.compression}"
        )

    # series 레벨별 shape 확인
    for i, level in enumerate(tif.series[0].levels):
        print(f"Level {i}: shape={level.shape}")

Page 0: shape=(10501, 16686, 3), dtype=uint8, is_tiled=True, tile=(256, 256), compression=8
Page 1: shape=(5250, 8343, 3), dtype=uint8, is_tiled=True, tile=(256, 256), compression=8
Page 2: shape=(2625, 4171, 3), dtype=uint8, is_tiled=True, tile=(256, 256), compression=8
Page 3: shape=(1312, 2085, 3), dtype=uint8, is_tiled=True, tile=(256, 256), compression=8
Page 4: shape=(656, 1042, 3), dtype=uint8, is_tiled=True, tile=(256, 256), compression=8
Page 5: shape=(328, 521, 3), dtype=uint8, is_tiled=True, tile=(256, 256), compression=8
Page 6: shape=(164, 260, 3), dtype=uint8, is_tiled=True, tile=(256, 256), compression=8
Page 7: shape=(82, 130, 3), dtype=uint8, is_tiled=True, tile=(256, 256), compression=8
Level 0: shape=(10501, 16686, 3)
Level 1: shape=(5250, 8343, 3)
Level 2: shape=(2625, 4171, 3)
Level 3: shape=(1312, 2085, 3)
Level 4: shape=(656, 1042, 3)
Level 5: shape=(328, 521, 3)
Level 6: shape=(164, 260, 3)
Level 7: shape=(82, 130, 3)


In [16]:

# OpenSlide 설치 확인
try:
    import openslide
    print('OpenSlide available:', openslide.__version__)
    slide = openslide.OpenSlide('/project_antwerp/hbae/data/Open_ST/GSM7990099_primary_HNSCC.tif')
    print('Dimensions:', slide.dimensions)
    print('Level count:', slide.level_count)
    print('Level dimensions:', slide.level_dimensions)
except ImportError:
    print('OpenSlide not available')

# pyvips 확인
try:
    import pyvips
    print('pyvips available:', pyvips.__version__)
except ImportError:
    print('pyvips not available')

# PIL 로 읽기 시도
from PIL import Image
try:
    img_pil = Image.open('/project_antwerp/hbae/data/Open_ST/GSM7990099_primary_HNSCC.tif')
    print('PIL format:', img_pil.format, img_pil.size, img_pil.mode)
except Exception as e:
    print('PIL error:', e)


OpenSlide not available
pyvips not available
PIL format: TIFF (16686, 10501) RGB


/opt/conda/lib/python3.11/site-packages/PIL/Image.py:3186: DecompressionBombWarning: Image size (175219686 pixels) exceeds limit of 89478485 pixels, could be decompression bomb DOS attack.
  warnings.warn(


In [17]:

import numpy as np
from PIL import Image
Image.MAX_IMAGE_PIXELS = None  # DecompressionBomb 경고 무시

img_pil = np.array(Image.open('/project_antwerp/hbae/data/Open_ST/GSM7990099_primary_HNSCC.tif'))
print('Shape:', img_pil.shape)

# tile 경계 discontinuity 재확인
print('=== PIL로 읽었을 때 tile boundary check ===')
for boundary in [256, 512, 768, 1024]:
    diff_row = abs(img_pil[boundary, 1000, 0].astype(int) - img_pil[boundary-1, 1000, 0].astype(int))
    diff_col = abs(img_pil[1000, boundary, 0].astype(int) - img_pil[1000, boundary-1, 0].astype(int))
    print(f'Row boundary {boundary}: pixel jump = {diff_row}')
    print(f'Col boundary {boundary}: pixel jump = {diff_col}')

diffs_normal = []
for r in range(300, 800, 10):
    if r % 256 != 0 and r % 256 != 255:
        diff = abs(img_pil[r, 1000, 0].astype(int) - img_pil[r-1, 1000, 0].astype(int))
        diffs_normal.append(diff)
print(f'Normal row diffs mean: {np.mean(diffs_normal):.2f}')


Shape: (10501, 16686, 3)
=== PIL로 읽었을 때 tile boundary check ===
Row boundary 256: pixel jump = 50
Col boundary 256: pixel jump = 18
Row boundary 512: pixel jump = 46
Col boundary 512: pixel jump = 30
Row boundary 768: pixel jump = 70
Col boundary 768: pixel jump = 15
Row boundary 1024: pixel jump = 27
Col boundary 1024: pixel jump = 40
Normal row diffs mean: 5.90


In [18]:

import numpy as np
import tifffile
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

img = tifffile.imread('/project_antwerp/hbae/data/Open_ST/GSM7990099_primary_HNSCC.tif')

# 전체 이미지를 작게 줄여서 grid artifact 패턴 확인
thumb = img[::10, ::10, :]
fig, ax = plt.subplots(1, 1, figsize=(12, 8))
ax.imshow(thumb)
# 256px 경계선 그리기 (원본 기준)
for x in range(0, img.shape[1], 256):
    ax.axvline(x=x//10, color='red', alpha=0.3, linewidth=0.5)
for y in range(0, img.shape[0], 256):
    ax.axhline(y=y//10, color='red', alpha=0.3, linewidth=0.5)
ax.set_title('Tile boundaries (red lines)')
plt.tight_layout()
plt.savefig('/project_antwerp/hbae/data/Open_ST/tile_boundary_check.png', dpi=100)
print('Saved')


Saved


In [1]:

from PIL import Image
import numpy as np

# 기존 Visium 패치 확인
img = Image.open('/project_antwerp/hbae/data/Processed_Data/GSE181300/GSM5494475/patches/AAACACCAATAACTGC-1.png')
print('Visium patch size:', img.size)
print('Visium patch mode:', img.mode)
arr = np.array(img)
print('Visium patch shape:', arr.shape)
print('Visium patch mean pixel:', arr.mean())
print('Visium patch min/max:', arr.min(), arr.max())


Visium patch size: (224, 224)
Visium patch mode: RGB
Visium patch shape: (224, 224, 3)
Visium patch mean pixel: 172.8732063137755
Visium patch min/max: 90 243


In [2]:

import numpy as np
import h5py
from PIL import Image
import os

# Open-ST 패치 10개를 PNG로 저장
out_dir = '/project_antwerp/hbae/data/Open_ST/patches_sample_png'
os.makedirs(out_dir, exist_ok=True)

with h5py.File('/project_antwerp/hbae/data/Open_ST/openst_patches.h5', 'r') as f:
    # 흰색 비율 낮은 (tissue 내부) 패치 찾기
    for i in range(len(f['patches'])):
        patch = f['patches'][i]
        white_ratio = (patch > 240).mean()
        if white_ratio < 0.05:
            img = Image.fromarray(patch)
            row = f['coords_row'][i]
            col = f['coords_col'][i]
            img.save(f'{out_dir}/cell_{i:05d}_r{row}_c{col}.png')
            print(f'Saved cell_{i:05d}: size={img.size}, mean={patch.mean():.1f}, white_ratio={white_ratio:.3f}')
            if len(os.listdir(out_dir)) >= 10:
                break

# Visium 패치랑 나란히 비교
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Visium 샘플
visium_patch = np.array(Image.open('/project_antwerp/hbae/data/Processed_Data/GSE181300/GSM5494475/patches/AAACACCAATAACTGC-1.png'))

fig, axes = plt.subplots(2, 5, figsize=(15, 7))
fig.suptitle('Top: Visium patches  |  Bottom: Open-ST patches', fontsize=12)

# Visium 5개
import glob
visium_files = glob.glob('/project_antwerp/hbae/data/Processed_Data/GSE181300/GSM5494475/patches/*.png')[:5]
for j, vf in enumerate(visium_files):
    vp = np.array(Image.open(vf))
    axes[0, j].imshow(vp)
    axes[0, j].set_title(f'Visium\n{vp.shape}\nmean={vp.mean():.0f}', fontsize=8)
    axes[0, j].axis('off')

# Open-ST 5개
openst_files = sorted(glob.glob(f'{out_dir}/*.png'))[:5]
for j, of in enumerate(openst_files):
    op = np.array(Image.open(of))
    axes[1, j].imshow(op)
    axes[1, j].set_title(f'Open-ST\n{op.shape}\nmean={op.mean():.0f}', fontsize=8)
    axes[1, j].axis('off')

plt.tight_layout()
plt.savefig('/project_antwerp/hbae/data/Open_ST/visium_vs_openst_patches.png', dpi=120)
print('Saved comparison figure')


Saved cell_00011: size=(224, 224), mean=96.1, white_ratio=0.009
Saved cell_00012: size=(224, 224), mean=102.5, white_ratio=0.027
Saved cell_00013: size=(224, 224), mean=95.9, white_ratio=0.004
Saved cell_00014: size=(224, 224), mean=92.8, white_ratio=0.000
Saved cell_00017: size=(224, 224), mean=92.1, white_ratio=0.000
Saved cell_00018: size=(224, 224), mean=95.7, white_ratio=0.000
Saved cell_00021: size=(224, 224), mean=87.7, white_ratio=0.000
Saved cell_00024: size=(224, 224), mean=93.4, white_ratio=0.000
Saved cell_00028: size=(224, 224), mean=90.9, white_ratio=0.000
Saved cell_00032: size=(224, 224), mean=86.6, white_ratio=0.000
Saved comparison figure


In [4]:

# Visium FOV 물리적 크기 재계산
# GSE181300 scalefactor 확인
import json, glob

sf_files = glob.glob('/project_antwerp/hbae/data/Processed_Data/GSE181300/GSM5494475/spatial/scalefactors_json.json')
if sf_files:
    with open(sf_files[0]) as f:
        sf = json.load(f)
    print('scalefactors:', sf)

# 원본 hires 이미지 크기 확인
from PIL import Image
import glob
hires_files = glob.glob('/project_antwerp/hbae/data/Processed_Data/GSE181300/GSM5494475/spatial/tissue_hires_image.png')
if hires_files:
    img = Image.open(hires_files[0])
    print('hires image size:', img.size)


In [5]:

# Visium과 동일한 물리 크기를 Open-ST에서 동일한 방식으로 재현하려면:
# Visium: 21px hires → 224px (10.7x upscale)
# Open-ST: 동일하게 작은 crop을 224로 upscale

# Open-ST에서 Visium과 동일한 물리 FOV(71µm)를 담는 pixel 수:
# 71 µm / 0.345 µm/px = 205.8 px → 약 206px 직접 crop 후 224로 resize

# 반면 Visium 학습과 완전히 동일한 시각적 스케일(blurry)을 재현하려면:
# Visium이 21px → 224px로 키운 것처럼, Open-ST도 동일 배율(10.7x)로
# 21px / 3.414 µm/px = 6.15 µm → Open-ST에서: 6.15 µm / 0.345 µm/px = 17.8px
# 즉 Open-ST에서 약 18px crop → 224px resize

import numpy as np
from PIL import Image
import h5py

# 방법 1: 206px crop (물리 FOV 매칭, 현재 방식과 유사)
# 방법 2: 18px crop → 224px resize (visual scale 매칭, Visium과 동일한 blurry 스타일)

# 방법 2 테스트
with h5py.File('/project_antwerp/hbae/data/Open_ST/openst_patches.h5', 'r') as f:
    patch_224 = f['patches'][14]   # tissue 내부 패치
    row = f['coords_row'][14]
    col = f['coords_col'][14]

import tifffile
from scipy.ndimage import zoom
img = tifffile.imread('/project_antwerp/hbae/data/Open_ST/GSM7990099_primary_HNSCC.tif')
mask = np.load('/project_antwerp/hbae/data/Open_ST/GSM7990099_primary_HNSCC/mask.npy')
H, W = img.shape[:2]
mask_full = zoom(mask, (H/mask.shape[0], W/mask.shape[1]), order=0).astype(np.uint8)
mask_3ch = np.stack([mask_full]*3, axis=-1)
img_masked = np.where(mask_3ch==1, img, 255).astype(np.uint8)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

# Visium 패치 (실제 학습 데이터)
vp = np.array(Image.open('/project_antwerp/hbae/data/Processed_Data/GSE181300/GSM5494475/patches/AAACACCAATAACTGC-1.png'))
axes[0].imshow(vp)
axes[0].set_title('Visium (학습)\n21px→224px resize\n71µm FOV', fontsize=9)
axes[0].axis('off')

# 현재 방식: 224px direct crop
axes[1].imshow(patch_224)
axes[1].set_title('Open-ST 현재\n224px direct crop\n77µm FOV', fontsize=9)
axes[1].axis('off')

# 방법 A: 206px crop → 224px resize (물리 FOV 일치)
half_206 = 103
r, c = int(row), int(col)
crop_206 = img_masked[r-half_206:r+half_206, c-half_206:c+half_206]
patch_206 = np.array(Image.fromarray(crop_206).resize((224,224), Image.BILINEAR))
axes[2].imshow(patch_206)
axes[2].set_title('Open-ST 방법 A\n206px→224px resize\n71µm FOV (물리일치)', fontsize=9)
axes[2].axis('off')

# 방법 B: 18px crop → 224px resize (visual scale 일치)
half_18 = 9
crop_18 = img_masked[r-half_18:r+half_18, c-half_18:c+half_18]
patch_18 = np.array(Image.fromarray(crop_18).resize((224,224), Image.BILINEAR))
axes[3].imshow(patch_18)
axes[3].set_title('Open-ST 방법 B\n18px→224px resize\n6µm FOV (visual일치)', fontsize=9)
axes[3].axis('off')

plt.suptitle('패치 추출 방식 비교', fontsize=12)
plt.tight_layout()
plt.savefig('/project_antwerp/hbae/data/Open_ST/patch_method_comparison.png', dpi=120)
print('Saved')


/tmp/ipykernel_346/771902681.py:70: UserWarning: Glyph 54617 (\N{HANGUL SYLLABLE HAG}) missing from current font.
  plt.tight_layout()
/tmp/ipykernel_346/771902681.py:70: UserWarning: Glyph 49845 (\N{HANGUL SYLLABLE SEUB}) missing from current font.
  plt.tight_layout()
/tmp/ipykernel_346/771902681.py:70: UserWarning: Glyph 54788 (\N{HANGUL SYLLABLE HYEON}) missing from current font.
  plt.tight_layout()
/tmp/ipykernel_346/771902681.py:70: UserWarning: Glyph 51116 (\N{HANGUL SYLLABLE JAE}) missing from current font.
  plt.tight_layout()
/tmp/ipykernel_346/771902681.py:70: UserWarning: Glyph 48169 (\N{HANGUL SYLLABLE BANG}) missing from current font.
  plt.tight_layout()
/tmp/ipykernel_346/771902681.py:70: UserWarning: Glyph 48277 (\N{HANGUL SYLLABLE BEOB}) missing from current font.
  plt.tight_layout()
/tmp/ipykernel_346/771902681.py:70: UserWarning: Glyph 47932 (\N{HANGUL SYLLABLE MUL}) missing from current font.
  plt.tight_layout()
/tmp/ipykernel_346/771902681.py:70: UserWarning: G

Saved


In [1]:

import numpy as np
import tifffile
from scipy.ndimage import zoom
from PIL import Image
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

img = tifffile.imread('/project_antwerp/hbae/data/Open_ST/GSM7990099_primary_HNSCC.tif')
mask = np.load('/project_antwerp/hbae/data/Open_ST/GSM7990099_primary_HNSCC/mask.npy')
H, W = img.shape[:2]
mask_full = zoom(mask, (H/mask.shape[0], W/mask.shape[1]), order=0).astype(np.uint8)
mask_3ch  = np.stack([mask_full]*3, axis=-1)
img_masked = np.where(mask_3ch==1, img, 255).astype(np.uint8)

# Visium과 동일한 해상도로 downsample
scale = 0.345 / 3.41   # Open-ST → Visium 해상도
print(f'Downsample scale: {scale:.4f} (1/{1/scale:.1f})')
img_down = zoom(img_masked, (scale, scale, 1), order=1)
print(f'Downsampled image: {img_down.shape}')

# downsampled 이미지에서 동일한 cell 좌표로 21px crop → 224px resize
r_orig, c_orig = 1740, 11564   # 아까 확인한 tissue 내부 cell
r_down = int(r_orig * scale)
c_down = int(c_orig * scale)

half = 10   # 21px / 2
crop_down = img_down[r_down-half:r_down+half, c_down-half:c_down+half]
patch_visium_style = np.array(Image.fromarray(crop_down).resize((224, 224), Image.BILINEAR))

# 비교
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

vp = np.array(Image.open('/project_antwerp/hbae/data/Processed_Data/GSE181300/GSM5494475/patches/AAACACCAATAACTGC-1.png'))
axes[0].imshow(vp)
axes[0].set_title(f'Visium 학습 패치\n3.41µm/px, 21px→224px\nmean={vp.mean():.0f}', fontsize=9)
axes[0].axis('off')

from h5py import File
with File('/project_antwerp/hbae/data/Open_ST/openst_patches.h5', 'r') as f:
    patch_current = f['patches'][17]
axes[1].imshow(patch_current)
axes[1].set_title(f'Open-ST 현재 방식\n0.345µm/px, 224px direct\nmean={patch_current.mean():.0f}', fontsize=9)
axes[1].axis('off')

axes[2].imshow(patch_visium_style)
axes[2].set_title(f'Open-ST downsample 방식\n3.41µm/px로 변환 후 21px→224px\nmean={patch_visium_style.mean():.0f}', fontsize=9)
axes[2].axis('off')

plt.suptitle('해상도 맞춤 비교', fontsize=12)
plt.tight_layout()
plt.savefig('/project_antwerp/hbae/data/Open_ST/resolution_match_comparison.png', dpi=120)
print('Saved')


Downsample scale: 0.1012 (1/9.9)
Downsampled image: (1062, 1688, 3)


/tmp/ipykernel_187/127780285.py:51: UserWarning: Glyph 54617 (\N{HANGUL SYLLABLE HAG}) missing from current font.
  plt.tight_layout()
/tmp/ipykernel_187/127780285.py:51: UserWarning: Glyph 49845 (\N{HANGUL SYLLABLE SEUB}) missing from current font.
  plt.tight_layout()
/tmp/ipykernel_187/127780285.py:51: UserWarning: Glyph 54056 (\N{HANGUL SYLLABLE PAE}) missing from current font.
  plt.tight_layout()
/tmp/ipykernel_187/127780285.py:51: UserWarning: Glyph 52824 (\N{HANGUL SYLLABLE CI}) missing from current font.
  plt.tight_layout()
/tmp/ipykernel_187/127780285.py:51: UserWarning: Glyph 54788 (\N{HANGUL SYLLABLE HYEON}) missing from current font.
  plt.tight_layout()
/tmp/ipykernel_187/127780285.py:51: UserWarning: Glyph 51116 (\N{HANGUL SYLLABLE JAE}) missing from current font.
  plt.tight_layout()
/tmp/ipykernel_187/127780285.py:51: UserWarning: Glyph 48169 (\N{HANGUL SYLLABLE BANG}) missing from current font.
  plt.tight_layout()
/tmp/ipykernel_187/127780285.py:51: UserWarning: Gly

Saved


In [2]:
# stain normalization 라이브러리 확인

try:
    import staintools
    print('staintools available')
except:
    print('staintools not available')

try:
    import torchstain
    print('torchstain available')
except:
    print('torchstain not available')

try:
    import spams
    print('spams available')
except:
    print('spams not available')


staintools not available
torchstain not available
spams not available


In [3]:

import tifffile
import numpy as np

with tifffile.TiffFile('/project_antwerp/hbae/data/Open_ST/GSM7990099_primary_HNSCC.tif') as tif:
    for i, level in enumerate(tif.series[0].levels):
        h, w = level.shape[0], level.shape[1]
        # 각 레벨의 실제 해상도 계산
        scale = h / 10501  # level 0 대비 비율
        um_per_px = 0.345 / scale
        print(f'Level {i}: {h:5d} x {w:5d}  |  {um_per_px:.3f} µm/px  |  scale={scale:.4f}')

# Visium hires = 3.41 µm/px 에 가장 가까운 레벨 찾기
print()
print('Visium hires: 3.41 µm/px')
print('가장 가까운 레벨을 쓰면 downsample 없이 바로 사용 가능!')


Level 0: 10501 x 16686  |  0.345 µm/px  |  scale=1.0000
Level 1:  5250 x  8343  |  0.690 µm/px  |  scale=0.5000
Level 2:  2625 x  4171  |  1.380 µm/px  |  scale=0.2500
Level 3:  1312 x  2085  |  2.761 µm/px  |  scale=0.1249
Level 4:   656 x  1042  |  5.523 µm/px  |  scale=0.0625
Level 5:   328 x   521  |  11.045 µm/px  |  scale=0.0312
Level 6:   164 x   260  |  22.091 µm/px  |  scale=0.0156
Level 7:    82 x   130  |  44.181 µm/px  |  scale=0.0078

Visium hires: 3.41 µm/px
가장 가까운 레벨을 쓰면 downsample 없이 바로 사용 가능!


In [4]:

import tifffile
import numpy as np
from PIL import Image
from scipy.ndimage import zoom
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import h5py

# Level 3 읽기
with tifffile.TiffFile('/project_antwerp/hbae/data/Open_ST/GSM7990099_primary_HNSCC.tif') as tif:
    img_l3 = tif.series[0].levels[3].asarray()
print(f'Level 3 shape: {img_l3.shape}')

# Level 3 해상도: 2.761 µm/px
# Visium FOV: 71 µm → 71/2.761 = 25.7px → round(1.3 × 25.7/1.3) 
# 즉 71µm / 2.761µm/px = 25.7px → 그냥 26px crop → 224px resize
um_per_px_l3 = 2.761
fov_um = 71.0
fov_px_l3 = round(fov_um / um_per_px_l3)
print(f'Level 3에서 FOV: {fov_px_l3} px')

# mask도 Level 3 기준으로 업샘플
mask = np.load('/project_antwerp/hbae/data/Open_ST/GSM7990099_primary_HNSCC/mask.npy')
H3, W3 = img_l3.shape[:2]
mask_l3 = zoom(mask, (H3/mask.shape[0], W3/mask.shape[1]), order=0).astype(np.uint8)
mask_3ch = np.stack([mask_l3]*3, axis=-1)
img_l3_masked = np.where(mask_3ch==1, img_l3, 255).astype(np.uint8)

# cell 좌표도 Level 3 스케일로 변환
scale_l3 = H3 / 10501
r_orig, c_orig = 1740, 11564
r_l3 = int(r_orig * scale_l3)
c_l3 = int(c_orig * scale_l3)
print(f'Original coord: ({r_orig}, {c_orig}) → Level3: ({r_l3}, {c_l3})')

half = fov_px_l3 // 2
crop = img_l3_masked[r_l3-half:r_l3+half, c_l3-half:c_l3+half]
patch_l3 = np.array(Image.fromarray(crop).resize((224, 224), Image.BILINEAR))

# 비교 시각화
fig, axes = plt.subplots(1, 3, figsize=(12, 4))

vp = np.array(Image.open('/project_antwerp/hbae/data/Processed_Data/GSE181300/GSM5494475/patches/AAACACCAATAACTGC-1.png'))
axes[0].imshow(vp)
axes[0].set_title(f'Visium 학습 패치\n3.41µm/px, 21px→224px\nmean={vp.mean():.0f}', fontsize=9)
axes[0].axis('off')

with h5py.File('/project_antwerp/hbae/data/Open_ST/openst_patches.h5', 'r') as f:
    patch_current = f['patches'][17]
axes[1].imshow(patch_current)
axes[1].set_title(f'Open-ST 현재 방식\n0.345µm/px, 224px direct\nmean={patch_current.mean():.0f}', fontsize=9)
axes[1].axis('off')

axes[2].imshow(patch_l3)
axes[2].set_title(f'Open-ST Level 3\n2.761µm/px, {fov_px_l3}px→224px\nmean={patch_l3.mean():.0f}', fontsize=9)
axes[2].axis('off')

plt.suptitle('해상도 맞춤 비교 (Pyramid TIFF Level 3 활용)', fontsize=12)
plt.tight_layout()
plt.savefig('/project_antwerp/hbae/data/Open_ST/level3_comparison.png', dpi=120)
print('Saved')
print(f'Level 3 patch mean: {patch_l3.mean():.1f}')


Level 3 shape: (1312, 2085, 3)
Level 3에서 FOV: 26 px
Original coord: (1740, 11564) → Level3: (217, 1444)


/tmp/ipykernel_187/233741759.py:60: UserWarning: Glyph 54617 (\N{HANGUL SYLLABLE HAG}) missing from current font.
  plt.tight_layout()
/tmp/ipykernel_187/233741759.py:60: UserWarning: Glyph 49845 (\N{HANGUL SYLLABLE SEUB}) missing from current font.
  plt.tight_layout()
/tmp/ipykernel_187/233741759.py:60: UserWarning: Glyph 54056 (\N{HANGUL SYLLABLE PAE}) missing from current font.
  plt.tight_layout()
/tmp/ipykernel_187/233741759.py:60: UserWarning: Glyph 52824 (\N{HANGUL SYLLABLE CI}) missing from current font.
  plt.tight_layout()
/tmp/ipykernel_187/233741759.py:60: UserWarning: Glyph 54788 (\N{HANGUL SYLLABLE HYEON}) missing from current font.
  plt.tight_layout()
/tmp/ipykernel_187/233741759.py:60: UserWarning: Glyph 51116 (\N{HANGUL SYLLABLE JAE}) missing from current font.
  plt.tight_layout()
/tmp/ipykernel_187/233741759.py:60: UserWarning: Glyph 48169 (\N{HANGUL SYLLABLE BANG}) missing from current font.
  plt.tight_layout()
/tmp/ipykernel_187/233741759.py:60: UserWarning: Gly

Saved
Level 3 patch mean: 91.6


In [5]:

import scanpy as sc
import numpy as np
import tifffile

adata = sc.read_h5ad('/project_antwerp/hbae/data/Open_ST/GSM7990099_primary_HNSCC.h5ad')
coords = adata.obsm['spatial']

# Level 0 이미지 크기
with tifffile.TiffFile('/project_antwerp/hbae/data/Open_ST/GSM7990099_primary_HNSCC.tif') as tif:
    l0 = tif.series[0].levels[0]
    l3 = tif.series[0].levels[3]
    print(f'Level 0: {l0.shape[0]} x {l0.shape[1]}')
    print(f'Level 3: {l3.shape[0]} x {l3.shape[1]}')

print(f'coords row range: {coords[:,0].min():.1f} ~ {coords[:,0].max():.1f}')
print(f'coords col range: {coords[:,1].min():.1f} ~ {coords[:,1].max():.1f}')

# Level 0 기준이면 coords max가 Level 0 크기 안에 있어야 함
print(f'Level 0 H={l0.shape[0]}, coords row max={coords[:,0].max():.1f} → ', end='')
print('Level 0 기준 ✓' if coords[:,0].max() < l0.shape[0] else '범위 초과!')

# Level 3 기준이면 coords max가 Level 3 크기 안에 있어야 함  
print(f'Level 3 H={l3.shape[0]}, coords row max={coords[:,0].max():.1f} → ', end='')
print('Level 3 기준 ✓' if coords[:,0].max() < l3.shape[0] else '범위 초과 → Level 0 기준')


Level 0: 10501 x 16686
Level 3: 1312 x 2085
coords row range: 1350.7 ~ 8830.4
coords col range: 3196.0 ~ 15042.0
Level 0 H=10501, coords row max=8830.4 → Level 0 기준 ✓
Level 3 H=1312, coords row max=8830.4 → 범위 초과 → Level 0 기준


In [6]:

import h5py
import numpy as np
from PIL import Image
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

with h5py.File('/project_antwerp/hbae/data/Open_ST/openst_patches_level3.h5', 'r') as f:
    # tissue 내부 패치 5개
    patches = f['patches'][:50:5]   # 0,5,10,15,20,25,30,35,40,45

fig, axes = plt.subplots(2, 5, figsize=(15, 7))
vp = np.array(Image.open('/project_antwerp/hbae/data/Processed_Data/GSE181300/GSM5494475/patches/AAACACCAATAACTGC-1.png'))

for j in range(5):
    axes[0, j].imshow(vp)
    axes[0, j].set_title(f'Visium\nmean={vp.mean():.0f}', fontsize=8)
    axes[0, j].axis('off')
    axes[1, j].imshow(patches[j])
    axes[1, j].set_title(f'Open-ST L3\nmean={patches[j].mean():.0f}', fontsize=8)
    axes[1, j].axis('off')

plt.suptitle('Visium vs Open-ST Level 3 patches', fontsize=12)
plt.tight_layout()
plt.savefig('/project_antwerp/hbae/data/Open_ST/final_patch_comparison.png', dpi=120)
print('Saved')


Saved


In [7]:

import h5py
import numpy as np
from PIL import Image
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

with h5py.File('/project_antwerp/hbae/data/Open_ST/openst_patches_level3.h5', 'r') as f:
    n = f.attrs['n_cells']
    # 흰색 비율 낮은 패치만 선택
    interior = []
    for i in range(n):
        p = f['patches'][i]
        white = (p > 240).mean()
        if white < 0.02:
            interior.append(i)
        if len(interior) >= 5:
            break
    patches = f['patches'][interior]
    print(f'Interior patches found at idx: {interior}')
    for i, p in enumerate(patches):
        print(f'  patch {interior[i]}: mean={p.mean():.1f}, white_ratio={(p>240).mean():.3f}')

vp = np.array(Image.open('/project_antwerp/hbae/data/Processed_Data/GSE181300/GSM5494475/patches/AAACACCAATAACTGC-1.png'))

fig, axes = plt.subplots(2, 5, figsize=(15, 7))
for j in range(5):
    axes[0, j].imshow(vp)
    axes[0, j].set_title(f'Visium\nmean={vp.mean():.0f}', fontsize=8)
    axes[0, j].axis('off')
    axes[1, j].imshow(patches[j])
    axes[1, j].set_title(f'Open-ST L3\nmean={patches[j].mean():.0f}', fontsize=8)
    axes[1, j].axis('off')

plt.suptitle('Interior cells only - Visium vs Open-ST Level 3', fontsize=12)
plt.tight_layout()
plt.savefig('/project_antwerp/hbae/data/Open_ST/interior_patch_comparison.png', dpi=120)
print('Saved')


Interior patches found at idx: [11, 13, 14, 17, 18]
  patch 11: mean=94.5, white_ratio=0.000
  patch 13: mean=96.5, white_ratio=0.000
  patch 14: mean=93.9, white_ratio=0.000
  patch 17: mean=91.6, white_ratio=0.000
  patch 18: mean=97.7, white_ratio=0.000
Saved


In [10]:

import numpy as np
import scanpy as sc
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

adata = sc.read_h5ad('/project_antwerp/hbae/data/Open_ST/GSM7990099_primary_HNSCC.h5ad')
coords = adata.obsm['spatial']  # Level 0 기준

# cell 간 거리 분포 확인 (샘플 1000개)
from sklearn.neighbors import KDTree
sample_idx = np.random.choice(len(coords), 1000, replace=False)
sample_coords = coords[sample_idx]

tree = KDTree(coords)
# 각 cell의 nearest neighbor 거리
dists, _ = tree.query(sample_coords, k=2)  # k=2 (자기 자신 제외)
nn_dists = dists[:, 1]

print(f'Cell간 nearest neighbor 거리 (Level 0 pixel 기준):')
print(f'  mean : {nn_dists.mean():.2f} px')
print(f'  median: {np.median(nn_dists):.2f} px')
print(f'  min  : {nn_dists.min():.2f} px')
print(f'  max  : {nn_dists.max():.2f} px')

# µm 변환
um = nn_dists * 0.345
print(f'Cell간 nearest neighbor 거리 (µm):')
print(f'  mean : {um.mean():.2f} µm')
print(f'  median: {np.median(um):.2f} µm')

# Visium spot diameter = 55µm → 반경 27.5µm
# 이 반경 안에 몇 개 cell이 있는지
VISIUM_RADIUS_UM = 27.5
VISIUM_RADIUS_PX = VISIUM_RADIUS_UM / 0.345
counts = tree.query_radius(coords, r=VISIUM_RADIUS_PX, count_only=True)
print(f'\nDistribution of the number of cells within the Visium spot radius ({VISIUM_RADIUS_UM}μm):')
print(f'  mean  : {counts.mean():.1f}')
print(f'  median: {np.median(counts):.1f}')
print(f'  min   : {counts.min()}')
print(f'  max   : {counts.max()}')
print(f'  distribution:')
for v in [1, 2, 3, 5, 10, 20, 50]:
    print(f'    >= {v:3d} cells: {(counts >= v).mean()*100:.1f}%')

# 히스토그램
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(nn_dists * 0.345, bins=50, color='steelblue', edgecolor='white')
axes[0].set_xlabel('Nearest neighbor distance (µm)')
axes[0].set_ylabel('Count')
axes[0].set_title('Close distance distribution between cells')
axes[0].axvline(VISIUM_RADIUS_UM, color='red', linestyle='--', label=f'Visium radius {VISIUM_RADIUS_UM}µm')
axes[0].legend()

axes[1].hist(counts, bins=50, color='coral', edgecolor='white')
axes[1].set_xlabel('Cells within Visium spot radius')
axes[1].set_ylabel('Count')
axes[1].set_title(f'Distribution of the number of cells within the Visium spot radius ({VISIUM_RADIUS_UM}μm)')
axes[1].axvline(counts.mean(), color='red', linestyle='--', label=f'mean={counts.mean():.1f}')
axes[1].legend()

plt.tight_layout()
plt.savefig('/project_antwerp/hbae/data/Open_ST/cell_distribution.png', dpi=120)
print('Saved')


Cell간 nearest neighbor 거리 (Level 0 pixel 기준):
  mean : 29.17 px
  median: 27.96 px
  min  : 16.07 px
  max  : 129.03 px
Cell간 nearest neighbor 거리 (µm):
  mean : 10.06 µm
  median: 9.65 µm

Distribution of the number of cells within the Visium spot radius (27.5μm):
  mean  : 13.4
  median: 14.0
  min   : 1
  max   : 28
  distribution:
    >=   1 cells: 100.0%
    >=   2 cells: 99.8%
    >=   3 cells: 99.4%
    >=   5 cells: 98.1%
    >=  10 cells: 84.2%
    >=  20 cells: 5.4%
    >=  50 cells: 0.0%
Saved


In [11]:

import scanpy as sc
import numpy as np

adata = sc.read_h5ad('/project_antwerp/hbae/data/Open_ST/GSM7990099_primary_HNSCC.h5ad')

print('obs columns:', list(adata.obs.columns))
print('uns keys:', list(adata.uns.keys()))
print('obsm keys:', list(adata.obsm.keys()))

# obs 첫 5행
print()
print(adata.obs.head())

# annotation 확인
if 'annotation' in adata.obs.columns:
    print('annotation unique:', adata.obs['annotation'].unique()[:10])
if 'tile' in str(adata.obs.columns).lower():
    print('tile 관련 컬럼 있음!')


obs columns: ['cell_ID_mask', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_reads', 'reads_per_counts', 'n_joined', 'exact_entropy', 'theoretical_entropy', 'exact_compression', 'theoretical_compression', 'n_counts', 'annotation']
uns keys: ['log1p', 'spatial']
obsm keys: ['spatial']

         cell_ID_mask  n_genes_by_counts  total_counts  total_counts_mt  \
cell_bc                                                                   
116               119                412         523.0             30.0   
139               143                469         589.0             36.0   
142               146                453         572.0             32.0   
144               148                485         622.0             32.0   
156               160                595         871.0             71.0   

         pct_counts_mt  n_reads  reads_per_counts  n_joined  exact_entropy  \
cell_bc                                                                      
116

In [13]:

import numpy as np
import scanpy as sc
from sklearn.neighbors import KDTree

adata = sc.read_h5ad('/project_antwerp/hbae/data/Open_ST/GSM7990099_primary_HNSCC.h5ad')
coords = adata.obsm['spatial']  # Level 0 기준, row/col

# 우리 패치 FOV = 71µm = 26px (Level 3) = 26 * (0.345/2.761) * (1/0.345) = 206px (Level 0)
FOV_RADIUS_UM = 71.0 / 2    # 35.5µm (패치 절반)
FOV_RADIUS_PX_L0 = FOV_RADIUS_UM / 0.345   # Level 0 기준

print(f'패치 FOV 반경: {FOV_RADIUS_UM} µm = {FOV_RADIUS_PX_L0:.1f} px (Level 0)')

# 각 cell 중심 기준으로 FOV 안에 몇 개 cell이 있는지
tree = KDTree(coords)
counts = tree.query_radius(coords, r=FOV_RADIUS_PX_L0, count_only=True)

print(f'\n패치 FOV 안의 cell 수 (자기 자신 포함):')
print(f'  mean  : {counts.mean():.1f}')
print(f'  median: {np.median(counts):.1f}')
print(f'  min   : {counts.min()}')
print(f'  max   : {counts.max()}')
print(f'  distribution:')
for thresh in [1, 3, 5, 10, 20, 50]:
    print(f'    >= {thresh:2d} cells: {(counts >= thresh).sum():5d} ({(counts >= thresh).mean()*100:.1f}%)')

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.hist(counts, bins=40, color='steelblue', edgecolor='white')
plt.axvline(counts.mean(), color='red', linestyle='--', label=f'mean={counts.mean():.1f}')
plt.xlabel('Number of cells within patch FOV (71µm)')
plt.ylabel('Count')
plt.title('Distribution of the number of cells in the patch FOV')
plt.legend()
plt.tight_layout()
plt.savefig('/project_antwerp/hbae/data/Open_ST/cells_per_patch.png', dpi=120)
print('Saved')


패치 FOV 반경: 35.5 µm = 102.9 px (Level 0)

패치 FOV 안의 cell 수 (자기 자신 포함):
  mean  : 21.7
  median: 22.0
  min   : 1
  max   : 43
  distribution:
    >=  1 cells: 36642 (100.0%)
    >=  3 cells: 36553 (99.8%)
    >=  5 cells: 36409 (99.4%)
    >= 10 cells: 35465 (96.8%)
    >= 20 cells: 24501 (66.9%)
    >= 50 cells:     0 (0.0%)
Saved


In [14]:

import numpy as np
import scanpy as sc
import h5py
from sklearn.neighbors import KDTree

# 전체 cell 좌표
adata = sc.read_h5ad('/project_antwerp/hbae/data/Open_ST/GSM7990099_primary_HNSCC.h5ad')
all_coords = adata.obsm['spatial']  # Level 0 기준 전체 cell

# h5에서 valid cell 좌표 (Step 1 필터링 후)
with h5py.File('/project_antwerp/hbae/data/Open_ST/openst_patches_level3.h5', 'r') as f:
    rows_l3 = f['coords_row'][:]
    cols_l3 = f['coords_col'][:]
    n = f.attrs['n_cells']
    scale = 1312 / 10501  # Level 3 scale

print(f'Valid cells in h5: {n}')

# h5 좌표를 Level 0으로 역변환
rows_l0 = rows_l3 / scale
cols_l0 = cols_l3 / scale
patch_coords_l0 = np.stack([rows_l0, cols_l0], axis=1)

# 전체 cell KDTree 구축
tree = KDTree(all_coords)

# 각 패치 중심 FOV 안의 cell 수
FOV_RADIUS_PX_L0 = (71.0 / 2) / 0.345
counts = tree.query_radius(patch_coords_l0, r=FOV_RADIUS_PX_L0, count_only=True)

print(f'Cells per patch FOV:')
print(f'  mean: {counts.mean():.1f}, median: {np.median(counts):.1f}')
for thresh in [5, 10, 15, 20]:
    valid = (counts >= thresh).sum()
    print(f'  >= {thresh} cells: {valid} patches ({valid/n*100:.1f}%)')


Valid cells in h5: 36034
Cells per patch FOV:
  mean: 21.8, median: 22.0
  >= 5 cells: 35858 patches (99.5%)
  >= 10 cells: 35072 patches (97.3%)
  >= 15 cells: 32336 patches (89.7%)
  >= 20 cells: 24437 patches (67.8%)


In [15]:

# Visium spot 안의 cell 수는 보통 문헌에서 알려져 있지만
# 우리 데이터에서 직접 확인해보자
# Open-ST cell density로 역산 가능

# Open-ST 기준:
# - cell 간 거리: mean 10µm
# - Visium spot radius 27.5µm 안에 mean 13.4개 cell
# - 우리 FOV(71µm diameter) 안에 mean 22개 cell

# Visium spot diameter = 55µm
# 문헌상 Visium spot당 cell 수: 1~10개 (tissue type에 따라)
# HNSCC같은 dense tumor는 보통 3~8개

# 실제로 계산해보면:
import numpy as np

# Open-ST cell density: cell 간 거리 10µm
# 세포 밀도 = 1 / (pi * r^2) where r = 10/2 = 5µm (근사)
cell_area_um2 = np.pi * (10/2)**2   # 한 세포당 차지하는 면적
visium_spot_area = np.pi * (27.5)**2  # Visium spot 면적

cells_per_visium = visium_spot_area / cell_area_um2
print(f'세포 면적(근사): {cell_area_um2:.1f} µm²')
print(f'Visium spot 면적: {visium_spot_area:.1f} µm²')
print(f'Visium spot당 예상 cell 수: {cells_per_visium:.1f}')

# 우리 FOV vs Visium spot
our_fov_area = np.pi * (71/2)**2
cells_per_fov = our_fov_area / cell_area_um2
print(f'우리 FOV 면적: {our_fov_area:.1f} µm²')
print(f'우리 FOV당 예상 cell 수: {cells_per_fov:.1f}')

print(f'우리 FOV가 Visium spot보다 {our_fov_area/visium_spot_area:.1f}배 더 큼')

# Visium spot 크기로 맞추려면 radius를 27.5µm로 줄여야 함
print(f'\n→ Visium spot과 맞추려면 FOV radius = 27.5µm 사용')
print(f'  현재 FOV radius = 35.5µm (71µm diameter)')
print(f'  Visium spot radius = 27.5µm (55µm diameter)')


세포 면적(근사): 78.5 µm²
Visium spot 면적: 2375.8 µm²
Visium spot당 예상 cell 수: 30.2
우리 FOV 면적: 3959.2 µm²
우리 FOV당 예상 cell 수: 50.4
우리 FOV가 Visium spot보다 1.7배 더 큼

→ Visium spot과 맞추려면 FOV radius = 27.5µm 사용
  현재 FOV radius = 35.5µm (71µm diameter)
  Visium spot radius = 27.5µm (55µm diameter)


In [16]:

try:
    import cellpose
    print('cellpose version:', cellpose.__version__)
except:
    print('cellpose not available')

try:
    import stardist
    print('stardist available')
except:
    print('stardist not available')


cellpose not available
stardist not available


In [19]:

import numpy as np
from PIL import Image
import glob
from scipy import ndimage

# Visium 패치 샘플 50장에서 핵 counting (Cellpose 없이)
# H&E에서 핵은 어두운 보라색 → threshold로 검출 가능
patch_files = glob.glob('/project_antwerp/hbae/data/Processed_Data/GSE181300/GSM5494477/patches/*.png')[:50]

nuclei_counts = []
for pf in patch_files:
    img = np.array(Image.open(pf))
    
    # H&E 핵 검출: Hematoxylin channel (어두운 보라)
    # R 낮고 B 높은 픽셀 = 핵
    r, g, b = img[:,:,0], img[:,:,1], img[:,:,2]
    
    # 핵 마스크: 어둡고 보라색인 영역
    nucleus_mask = (r < 160) & (b > r) & (img.mean(axis=2) < 180)
    
    # connected components로 핵 개수 추정
    labeled, n_nuclei = ndimage.label(nucleus_mask)
    
    # 너무 작은 것(noise) 제거
    sizes = ndimage.sum(nucleus_mask, labeled, range(1, n_nuclei+1))
    valid_nuclei = sum(1 for s in sizes if s > 30)  # 30px² 이상
    nuclei_counts.append(valid_nuclei)

nuclei_counts = np.array(nuclei_counts)
print(f'Visium 패치당 핵 수 (50장 샘플):')
print(f'  mean  : {nuclei_counts.mean():.1f}')
print(f'  median: {np.median(nuclei_counts):.1f}')
print(f'  min   : {nuclei_counts.min()}')
print(f'  max   : {nuclei_counts.max()}')
print(f'  std   : {nuclei_counts.std():.1f}')


Visium 패치당 핵 수 (50장 샘플):
  mean  : 13.8
  median: 13.0
  min   : 0
  max   : 35
  std   : 9.1


In [ ]:

import numpy as np
from PIL import Image
import glob
from scipy import ndimage
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os

# 모든 GSM 샘플의 패치 폴더 찾기
patch_dirs = glob.glob('/project_antwerp/hbae/data/Processed_Data/*/*/patches/')
print(f'Found {len(patch_dirs)} samples')

all_counts = []
sample_means = []

for patch_dir in sorted(patch_dirs):
    sample = patch_dir.split('/')[-3]  # GSM ID
    patch_files = glob.glob(f'{patch_dir}*.png')
    
    if len(patch_files) == 0:
        continue
    
    sample_counts = []
    for pf in patch_files:
        try:
            img = np.array(Image.open(pf))
            r, g, b = img[:,:,0], img[:,:,1], img[:,:,2]
            nucleus_mask = (r < 160) & (b > r) & (img.mean(axis=2) < 180)
            labeled, n = ndimage.label(nucleus_mask)
            sizes = ndimage.sum(nucleus_mask, labeled, range(1, n+1))
            valid = sum(1 for s in sizes if s > 30)
            sample_counts.append(valid)
        except:
            continue
    
    if sample_counts:
        sample_arr = np.array(sample_counts)
        all_counts.extend(sample_counts)
        sample_means.append({'sample': sample, 'mean': sample_arr.mean(), 
                             'median': np.median(sample_arr), 'n': len(sample_counts)})
        print(f'  {sample}: {len(sample_counts)} patches, mean={sample_arr.mean():.1f}')

all_counts = np.array(all_counts)
print(f'\n전체 {len(all_counts)} patches:')
print(f'  mean  : {all_counts.mean():.1f}')
print(f'  median: {np.median(all_counts):.1f}')
print(f'  std   : {all_counts.std():.1f}')
print(f'  min   : {all_counts.min()}')
print(f'  max   : {all_counts.max()}')

# 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 전체 distribution
axes[0].hist(all_counts, bins=40, color='steelblue', edgecolor='white')
axes[0].axvline(all_counts.mean(), color='red', linestyle='--', 
                label=f'mean={all_counts.mean():.1f}')
axes[0].axvline(21.7, color='orange', linestyle='--',
                label='Open-ST FOV mean=21.7')
axes[0].set_xlabel('Nuclei per Visium patch')
axes[0].set_ylabel('Count')
axes[0].set_title(f'전체 Visium 패치 핵 수 분포\n(n={len(all_counts):,} patches)')
axes[0].legend()

# 샘플별 mean
import pandas as pd
df = pd.DataFrame(sample_means).sort_values('mean')
axes[1].barh(df['sample'], df['mean'], color='coral', edgecolor='white')
axes[1].axvline(21.7, color='orange', linestyle='--', label='Open-ST FOV mean=21.7')
axes[1].axvline(all_counts.mean(), color='red', linestyle='--', 
                label=f'Visium mean={all_counts.mean():.1f}')
axes[1].set_xlabel('Mean nuclei per patch')
axes[1].set_title('샘플별 평균 핵 수')
axes[1].legend(fontsize=7)

plt.tight_layout()
plt.savefig('/project_antwerp/hbae/data/Open_ST/visium_nuclei_per_patch.png', dpi=120)
print('Saved')


Found 36 samples
  GSM5494475: 1153 patches, mean=17.1
  GSM5494476: 1724 patches, mean=18.8
  GSM5494477: 1189 patches, mean=15.5
  GSM5494478: 1521 patches, mean=14.3
  GSM6339631_s1: 1159 patches, mean=1.6
  GSM6339632_s2: 1776 patches, mean=1.5
  GSM6339633_s3: 963 patches, mean=2.1
  GSM6339634_s4: 1946 patches, mean=1.6
  GSM6339635_s5: 1673 patches, mean=7.5
  GSM6339637_s7: 2490 patches, mean=8.4
  GSM6339638_s8: 2383 patches, mean=9.5
  GSM6339640_s10: 2700 patches, mean=9.9
  GSM6339641_s11: 1973 patches, mean=6.9
  GSM6339642_s12: 1497 patches, mean=6.6
  Patient1: 4195 patches, mean=10.7
  Patient2: 4287 patches, mean=11.1
  Patient3: 4478 patches, mean=12.2
  Patient4: 4293 patches, mean=12.3
  GSM7998252: 11924 patches, mean=0.8
  GSM7998253: 11915 patches, mean=1.0
  GSM7998254: 11919 patches, mean=1.3
  GSM7998255: 11914 patches, mean=0.8
  GSM7998256: 11913 patches, mean=1.4


In [22]:

import numpy as np
from PIL import Image
import glob
from scipy import ndimage
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os

# 이미 완료된 샘플 제외
done_samples = {
    'GSM5494475','GSM5494476','GSM5494477','GSM5494478',
    'GSM6339631_s1','GSM6339632_s2','GSM6339633_s3','GSM6339634_s4',
    'GSM6339635_s5','GSM6339637_s7','GSM6339638_s8','GSM6339640_s10',
    'GSM6339641_s11','GSM6339642_s12',
    'Patient1','Patient2','Patient3','Patient4',
    'GSM7998252','GSM7998253','GSM7998254','GSM7998255','GSM7998256'
}

patch_dirs = glob.glob('/project_antwerp/hbae/data/Processed_Data/*/*/patches/')
todo_dirs  = [d for d in sorted(patch_dirs)
              if d.split('/')[-3] not in done_samples]
print(f'Remaining samples: {len(todo_dirs)}')
for d in todo_dirs:
    print(f"  {d.split('/')[-3]}")

# 이미 완료된 결과 (수동 입력)
already_done = {
    'GSM5494475':  {'mean':17.1,'n':1153},
    'GSM5494476':  {'mean':18.8,'n':1724},
    'GSM5494477':  {'mean':15.5,'n':1189},
    'GSM5494478':  {'mean':14.3,'n':1521},
    'GSM6339631_s1': {'mean':1.6,'n':1159},
    'GSM6339632_s2': {'mean':1.5,'n':1776},
    'GSM6339633_s3': {'mean':2.1,'n':963},
    'GSM6339634_s4': {'mean':1.6,'n':1946},
    'GSM6339635_s5': {'mean':7.5,'n':1673},
    'GSM6339637_s7': {'mean':8.4,'n':2490},
    'GSM6339638_s8': {'mean':9.5,'n':2383},
    'GSM6339640_s10':{'mean':9.9,'n':2700},
    'GSM6339641_s11':{'mean':6.9,'n':1973},
    'GSM6339642_s12':{'mean':6.6,'n':1497},
    'Patient1':    {'mean':10.7,'n':4195},
    'Patient2':    {'mean':11.1,'n':4287},
    'Patient3':    {'mean':12.2,'n':4478},
    'Patient4':    {'mean':12.3,'n':4293},
    'GSM7998252':  {'mean':0.8,'n':11924},
    'GSM7998253':  {'mean':1.0,'n':11915},
    'GSM7998254':  {'mean':1.3,'n':11919},
    'GSM7998255':  {'mean':0.8,'n':11914},
    'GSM7998256':  {'mean':1.4,'n':11913},
}

# 나머지 샘플 처리
new_results = {}
for patch_dir in todo_dirs:
    sample = patch_dir.split('/')[-3]
    patch_files = glob.glob(f'{patch_dir}*.png')
    if not patch_files:
        continue
    
    sample_counts = []
    for pf in patch_files:
        try:
            img = np.array(Image.open(pf))
            r, g, b = img[:,:,0], img[:,:,1], img[:,:,2]
            nucleus_mask = (r < 160) & (b > r) & (img.mean(axis=2) < 180)
            labeled, n = ndimage.label(nucleus_mask)
            sizes = ndimage.sum(nucleus_mask, labeled, range(1, n+1))
            valid = sum(1 for s in sizes if s > 30)
            sample_counts.append(valid)
        except:
            continue
    
    if sample_counts:
        arr = np.array(sample_counts)
        new_results[sample] = {'mean': arr.mean(), 'n': len(sample_counts)}
        print(f'  {sample}: {len(sample_counts)} patches, mean={arr.mean():.1f}')

# 전체 합치기
all_results = {**already_done, **new_results}
print(f'\nTotal samples: {len(all_results)}')

# weighted mean (패치 수 가중)
total_weighted = sum(v['mean'] * v['n'] for v in all_results.values())
total_n = sum(v['n'] for v in all_results.values())
weighted_mean = total_weighted / total_n
print(f'Weighted mean nuclei per patch: {weighted_mean:.1f}')
print(f'Open-ST FOV mean: 21.7')

# 시각화
import pandas as pd
df = pd.DataFrame([{'sample': k, **v} for k, v in all_results.items()])
df = df.sort_values('mean')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 샘플별 bar
colors = ['steelblue' if m > 5 else 'lightcoral' 
          for m in df['mean']]
axes[0].barh(df['sample'], df['mean'], color=colors, edgecolor='white')
axes[0].axvline(weighted_mean, color='red', linestyle='--',
                label=f'Weighted mean={weighted_mean:.1f}')
axes[0].axvline(21.7, color='orange', linestyle='--',
                label='Open-ST FOV=21.7')
axes[0].set_xlabel('Mean nuclei per patch')
axes[0].set_title('샘플별 Visium 패치당 평균 핵 수')
axes[0].legend()

# 정상 샘플만 (mean > 5) distribution
normal = df[df['mean'] > 5]
axes[1].bar(normal['sample'], normal['mean'], color='steelblue', edgecolor='white')
axes[1].axhline(weighted_mean, color='red', linestyle='--',
                label=f'Weighted mean={weighted_mean:.1f}')
axes[1].axhline(21.7, color='orange', linestyle='--',
                label='Open-ST FOV=21.7')
axes[1].set_xticklabels(normal['sample'], rotation=45, ha='right', fontsize=7)
axes[1].set_ylabel('Mean nuclei per patch')
axes[1].set_title('정상 샘플만 (mean > 5)')
axes[1].legend()

plt.tight_layout()
plt.savefig('/project_antwerp/hbae/data/Open_ST/visium_nuclei_per_patch_full.png', dpi=120)
print('Saved')
np.save('/project_antwerp/hbae/data/Open_ST/visium_nuclei_results.npy', all_results)


Remaining samples: 13
  GSM7998257
  GSM7998258
  GSM7998259
  GSM8633891_21_00757_LI_SING
  GSM8633892_21_00758_LI_SING
  GSM8633893_21_01569_LI_SING
  GSM8633894_21_01570_LI_SING
  GSM8633895_21_01586_LI_SING
  GSM8633896_21_01587_LI_SING
  P5
  Visium_S01
  17B5776
  19h1257
  GSM7998257: 11913 patches, mean=1.7
  GSM7998258: 11916 patches, mean=0.7
  GSM7998259: 11915 patches, mean=1.3
  GSM8633891_21_00757_LI_SING: 1186 patches, mean=9.8
  GSM8633892_21_00758_LI_SING: 1578 patches, mean=11.2
  GSM8633893_21_01569_LI_SING: 1359 patches, mean=3.1
  GSM8633894_21_01570_LI_SING: 1897 patches, mean=2.4
  GSM8633895_21_01586_LI_SING: 1194 patches, mean=8.9
  GSM8633896_21_01587_LI_SING: 1437 patches, mean=7.7
  P5: 2873 patches, mean=5.9
  Visium_S01: 2077 patches, mean=11.3
  17B5776: 2232 patches, mean=8.1
  19h1257: 2226 patches, mean=0.0

Total samples: 36
Weighted mean nuclei per patch: 4.1
Open-ST FOV mean: 21.7


/tmp/ipykernel_187/3812685018.py:117: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  axes[1].set_xticklabels(normal['sample'], rotation=45, ha='right', fontsize=7)
/tmp/ipykernel_187/3812685018.py:122: UserWarning: Glyph 49368 (\N{HANGUL SYLLABLE SAEM}) missing from current font.
  plt.tight_layout()
/tmp/ipykernel_187/3812685018.py:122: UserWarning: Glyph 54540 (\N{HANGUL SYLLABLE PEUL}) missing from current font.
  plt.tight_layout()
/tmp/ipykernel_187/3812685018.py:122: UserWarning: Glyph 48324 (\N{HANGUL SYLLABLE BYEOL}) missing from current font.
  plt.tight_layout()
/tmp/ipykernel_187/3812685018.py:122: UserWarning: Glyph 54056 (\N{HANGUL SYLLABLE PAE}) missing from current font.
  plt.tight_layout()
/tmp/ipykernel_187/3812685018.py:122: UserWarning: Glyph 52824 (\N{HANGUL SYLLABLE CI}) missing from current font.
  plt.tight_layout()
/tmp/ipykernel_187/3812685018.py:122: UserWarning: Glyph 45817 (\

Saved


In [24]:

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# 제외할 샘플
exclude = {
    'GSM7998252','GSM7998253','GSM7998254','GSM7998255',
    'GSM7998256','GSM7998257','GSM7998258','GSM7998259',
    'Patient1','Patient2','Patient3','Patient4','19h1257'
}

all_results = {
    'GSM5494475':  {'mean':17.1,'n':1153},
    'GSM5494476':  {'mean':18.8,'n':1724},
    'GSM5494477':  {'mean':15.5,'n':1189},
    'GSM5494478':  {'mean':14.3,'n':1521},
    'GSM6339631_s1': {'mean':1.6,'n':1159},
    'GSM6339632_s2': {'mean':1.5,'n':1776},
    'GSM6339633_s3': {'mean':2.1,'n':963},
    'GSM6339634_s4': {'mean':1.6,'n':1946},
    'GSM6339635_s5': {'mean':7.5,'n':1673},
    'GSM6339637_s7': {'mean':8.4,'n':2490},
    'GSM6339638_s8': {'mean':9.5,'n':2383},
    'GSM6339640_s10':{'mean':9.9,'n':2700},
    'GSM6339641_s11':{'mean':6.9,'n':1973},
    'GSM6339642_s12':{'mean':6.6,'n':1497},
    'GSM8633891_21_00757_LI_SING': {'mean':9.8,'n':1186},
    'GSM8633892_21_00758_LI_SING': {'mean':11.2,'n':1578},
    'GSM8633893_21_01569_LI_SING': {'mean':3.1,'n':1359},
    'GSM8633894_21_01570_LI_SING': {'mean':2.4,'n':1897},
    'GSM8633895_21_01586_LI_SING': {'mean':8.9,'n':1194},
    'GSM8633896_21_01587_LI_SING': {'mean':7.7,'n':1437},
    'P5':         {'mean':5.9,'n':2873},
    'Visium_S01': {'mean':11.3,'n':2077},
    '17B5776':    {'mean':8.1,'n':2232},
}

# 제외 후 필터
valid = {k:v for k,v in all_results.items() if k not in exclude}
print(f'Valid samples: {len(valid)}')

# 각 샘플 정보
import pandas as pd
df = pd.DataFrame([{'sample':k, **v} for k,v in valid.items()])
df = df.sort_values('mean')

# weighted mean
total_w = sum(v['mean']*v['n'] for v in valid.values())
total_n = sum(v['n'] for v in valid.values())
w_mean  = total_w / total_n
print(f'Weighted mean nuclei per patch: {w_mean:.1f}')
print(f'Open-ST FOV mean             : 21.7')

# dataset별 색상 구분
dataset_colors = {}
for s in df['sample']:
    if s.startswith('GSM5494'):
        dataset_colors[s] = '#2196F3'  # GSE181300 파랑
    elif s.startswith('GSM6339'):
        dataset_colors[s] = '#4CAF50'  # GSE208253 초록
    elif s.startswith('GSM8633'):
        dataset_colors[s] = '#FF9800'  # GSE281978 주황
    else:
        dataset_colors[s] = '#9C27B0'  # Queensland/Zenodo 보라

colors = [dataset_colors[s] for s in df['sample']]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 샘플별 bar (색상 구분)
bars = axes[0].barh(df['sample'], df['mean'], color=colors, edgecolor='white')
axes[0].axvline(w_mean, color='red', linestyle='--',
                label=f'Weighted mean={w_mean:.1f}')
axes[0].axvline(21.7, color='orange', linestyle='--',
                label='Open-ST FOV mean=21.7')
axes[0].set_xlabel('Mean nuclei per patch')
axes[0].set_title('샘플별 Visium 패치당 평균 핵 수\n(제외 샘플 제거 후)')
# legend for datasets
from matplotlib.patches import Patch
legend_elements = [
    Patch(color='#2196F3', label='GSE181300'),
    Patch(color='#4CAF50', label='GSE208253'),
    Patch(color='#FF9800', label='GSE281978'),
    Patch(color='#9C27B0', label='Queensland/Zenodo'),
    plt.Line2D([0],[0], color='red', linestyle='--', label=f'Weighted mean={w_mean:.1f}'),
    plt.Line2D([0],[0], color='orange', linestyle='--', label='Open-ST FOV=21.7'),
]
axes[0].legend(handles=legend_elements, fontsize=7)

# histogram (전체 분포)
means = df['mean'].values
axes[1].hist(means, bins=15, color='steelblue', edgecolor='white')
axes[1].axvline(w_mean, color='red', linestyle='--',
                label=f'Weighted mean={w_mean:.1f}')
axes[1].axvline(21.7, color='orange', linestyle='--',
                label='Open-ST FOV mean=21.7')
axes[1].set_xlabel('Mean nuclei per patch')
axes[1].set_ylabel('Number of samples')
axes[1].set_title('샘플 평균 핵 수 분포')
axes[1].legend()

plt.tight_layout()
plt.savefig('/project_antwerp/hbae/data/Open_ST/visium_nuclei_valid_samples.png', dpi=120)
print('Saved')

# 상세 출력
print(f'\n샘플별 상세:')
for _, row in df.sort_values('mean', ascending=False).iterrows():
    print(f"  {row['sample']:40s}: mean={row['mean']:5.1f}, n={row['n']:5d}")


Valid samples: 23
Weighted mean nuclei per patch: 8.2
Open-ST FOV mean             : 21.7


/tmp/ipykernel_187/2238731455.py:103: UserWarning: Glyph 49368 (\N{HANGUL SYLLABLE SAEM}) missing from current font.
  plt.tight_layout()
/tmp/ipykernel_187/2238731455.py:103: UserWarning: Glyph 54540 (\N{HANGUL SYLLABLE PEUL}) missing from current font.
  plt.tight_layout()
/tmp/ipykernel_187/2238731455.py:103: UserWarning: Glyph 48324 (\N{HANGUL SYLLABLE BYEOL}) missing from current font.
  plt.tight_layout()
/tmp/ipykernel_187/2238731455.py:103: UserWarning: Glyph 54056 (\N{HANGUL SYLLABLE PAE}) missing from current font.
  plt.tight_layout()
/tmp/ipykernel_187/2238731455.py:103: UserWarning: Glyph 52824 (\N{HANGUL SYLLABLE CI}) missing from current font.
  plt.tight_layout()
/tmp/ipykernel_187/2238731455.py:103: UserWarning: Glyph 45817 (\N{HANGUL SYLLABLE DANG}) missing from current font.
  plt.tight_layout()
/tmp/ipykernel_187/2238731455.py:103: UserWarning: Glyph 54217 (\N{HANGUL SYLLABLE PYEONG}) missing from current font.
  plt.tight_layout()
/tmp/ipykernel_187/2238731455.py:1

Saved

샘플별 상세:
  GSM5494476                              : mean= 18.8, n= 1724
  GSM5494475                              : mean= 17.1, n= 1153
  GSM5494477                              : mean= 15.5, n= 1189
  GSM5494478                              : mean= 14.3, n= 1521
  Visium_S01                              : mean= 11.3, n= 2077
  GSM8633892_21_00758_LI_SING             : mean= 11.2, n= 1578
  GSM6339640_s10                          : mean=  9.9, n= 2700
  GSM8633891_21_00757_LI_SING             : mean=  9.8, n= 1186
  GSM6339638_s8                           : mean=  9.5, n= 2383
  GSM8633895_21_01586_LI_SING             : mean=  8.9, n= 1194
  GSM6339637_s7                           : mean=  8.4, n= 2490
  17B5776                                 : mean=  8.1, n= 2232
  GSM8633896_21_01587_LI_SING             : mean=  7.7, n= 1437
  GSM6339635_s5                           : mean=  7.5, n= 1673
  GSM6339641_s11                          : mean=  6.9, n= 1973
  GSM6339642_s12         

In [25]:

import numpy as np

train_exprs = np.load('/project_antwerp/hbae/Loki_output/0317_epoch10_finetune_embedding_new/fold_01/train_exprs.npy')
print('train_exprs shape:', train_exprs.shape)
print('train_exprs min/max/mean:', train_exprs.min(), train_exprs.max(), train_exprs.mean())
print('nonzero ratio:', (train_exprs > 0).mean())

# Open-ST GT expression 확인
import h5py
with h5py.File('/project_antwerp/hbae/data/Open_ST/openst_patches_agg_mc10.h5', 'r') as f:
    gt_exprs = f['expression'][:100]
print('GT exprs min/max/mean:', gt_exprs.min(), gt_exprs.max(), gt_exprs.mean())
print('GT nonzero ratio:', (gt_exprs > 0).mean())


train_exprs shape: (49156, 2000)
train_exprs min/max/mean: 0.0 8.225646 0.28990695
nonzero ratio: 0.24931806900480105
GT exprs min/max/mean: 0.0 6.2955427 0.6750755
GT nonzero ratio: 0.29558067831449125


In [26]:

import numpy as np

# train_exprs가 어떻게 만들어졌는지 확인
# save_embeddings.py에서 gt_expr은 combined_expression_matrix.npy
gt_expr_visium = np.load('/project_antwerp/hbae/data/0317_HVG_NEW/combined_expression_matrix.npy')
print('Visium combined_expression_matrix:')
print('  shape:', gt_expr_visium.shape)
print('  min/max/mean:', gt_expr_visium.min(), gt_expr_visium.max(), gt_expr_visium.mean())
print('  nonzero ratio:', (gt_expr_visium > 0).mean())

# 첫 번째 spot 확인
print('  first spot sum:', gt_expr_visium[0].sum())
print('  first spot nonzero:', (gt_expr_visium[0] > 0).sum())


Visium combined_expression_matrix:
  shape: (51893, 17368)
  min/max/mean: 0.0 8.225646 0.16036716
  nonzero ratio: 0.18123674620374244
  first spot sum: 2278.5168
  first spot nonzero: 1406


In [27]:

import numpy as np

gt_expr = np.load('/project_antwerp/hbae/data/0317_HVG_NEW/combined_expression_matrix.npy')

# spot 하나 분석
spot = gt_expr[0]
nonzero = spot[spot > 0]

print('First spot 분석:')
print(f'  sum of log values: {spot.sum():.2f}')
print(f'  nonzero count: {(spot>0).sum()}')
print(f'  max: {spot.max():.4f}')

# log1p 역변환해서 원본 값 추정
raw = np.expm1(spot)
print(f'  expm1 후 sum (= normalize된 counts 합): {raw.sum():.1f}')
print(f'  expm1 후 max: {raw.max():.1f}')

# normalize_total target 추정
# normalize_total은 각 spot의 counts를 target_sum으로 맞춤
# expm1 후 sum이 target_sum * n_genes_nonzero/n_genes_total 아님
# 그냥 expm1 후 sum을 보면 target_sum 알 수 있음
print(f'  → target_sum 추정: {raw.sum():.0f}')

# 여러 spot 확인
sums = []
for i in range(100):
    s = np.expm1(gt_expr[i]).sum()
    sums.append(s)
sums = np.array(sums)
print(f'\n100개 spot expm1 sum:')
print(f'  mean: {sums.mean():.1f}')
print(f'  std : {sums.std():.1f}')
print(f'  → normalize_total target_sum ≈ {sums.mean():.0f}')

# Open-ST GT와 비교
import h5py
with h5py.File('/project_antwerp/hbae/data/Open_ST/openst_patches_agg_mc10.h5', 'r') as f:
    gt_openst = f['expression'][:100]

sums_openst = []
for i in range(100):
    s = np.expm1(gt_openst[i]).sum()
    sums_openst.append(s)
sums_openst = np.array(sums_openst)
print(f'\nOpen-ST GT expm1 sum:')
print(f'  mean: {sums_openst.mean():.1f}')
print(f'  std : {sums_openst.std():.1f}')
print(f'  → normalize_total target_sum ≈ {sums_openst.mean():.0f}')


First spot 분석:
  sum of log values: 2278.52
  nonzero count: 1406
  max: 5.1881
  expm1 후 sum (= normalize된 counts 합): 6962.5
  expm1 후 max: 178.1
  → target_sum 추정: 6962

100개 spot expm1 sum:
  mean: 6959.0
  std : 405.4
  → normalize_total target_sum ≈ 6959

Open-ST GT expm1 sum:
  mean: 10000.0
  std : 0.0
  → normalize_total target_sum ≈ 10000


In [28]:

import numpy as np

# combined_expression_matrix가 어떻게 만들어졌는지
# HVG 2000개만 subset됐으니까 전체 gene 기준 normalize 후 HVG subset한 것
# 전체 gene normalize_total(1e4) 후 HVG 2000개만 남기면
# HVG 2000개의 합이 전체의 일부만 됨

# HVG 2000개가 전체 발현의 몇 % 차지하는지 추정
# expm1 sum = 6959 이면 HVG가 전체의 69.6% 차지
print(f'HVG 2000개가 전체 발현의 {6959/10000*100:.1f}% 차지 (추정)')

# 즉 Visium pipeline:
# 1. 전체 gene normalize_total(target=10000)
# 2. log1p
# 3. HVG 2000개만 subset → sum이 ~7000으로 줄어듦

# Open-ST pipeline (현재):
# 1. HVG 1946개만 먼저 subset
# 2. normalize_total(target=10000)  ← 이게 문제!
# 3. log1p → sum이 항상 10000

print('문제: Open-ST는 HVG subset 후 normalize → target=10000')
print('정답: 전체 gene normalize 후 HVG subset → target~7000')


HVG 2000개가 전체 발현의 69.6% 차지 (추정)
문제: Open-ST는 HVG subset 후 normalize → target=10000
정답: 전체 gene normalize 후 HVG subset → target~7000


In [29]:
# 현재 h5 파일로 scale 확인만 먼저 해보자

import numpy as np, h5py

with h5py.File('/project_antwerp/hbae/data/Open_ST/openst_patches_agg_mc10.h5', 'r') as f:
    expr = f['expression'][:100]

sums = np.expm1(expr).sum(axis=1)
print(f'expm1 sum mean: {sums.mean():.0f}')
print(f'Visium 기준: ~6959')
print(f'맞으면 OK, 10000이면 수정 필요')


expm1 sum mean: 10000
Visium 기준: ~6959
맞으면 OK, 10000이면 수정 필요


In [30]:

import numpy as np
import scanpy as sc

adata = sc.read_h5ad('/project_antwerp/hbae/data/Open_ST/GSM7990099_primary_HNSCC.h5ad')
raw = adata.layers['raw']
import scipy.sparse as sp
if sp.issparse(raw): raw = raw.toarray()

# HVG gene index
hvg_genes = open('/project_antwerp/hbae/data/0317_hvg_2000_list.txt').read().strip().split('\n')
gene_names = list(adata.var_names)
gene_to_idx = {g: i for i, g in enumerate(gene_names)}
common_genes = [g for g in hvg_genes if g in gene_to_idx]
hvg_col_idx = [gene_to_idx[g] for g in common_genes]

# 전체 gene 중 HVG가 차지하는 비율
spot = raw[0]
total = spot.sum()
hvg_sum = spot[hvg_col_idx].sum()
print(f'Open-ST 첫 cell: total counts={total:.0f}, HVG counts={hvg_sum:.0f}')
print(f'HVG 비율: {hvg_sum/total*100:.1f}%')

# 전체 cell 평균
hvg_ratios = raw[:, hvg_col_idx].sum(axis=1) / raw.sum(axis=1)
print(f'전체 cell 평균 HVG 비율: {hvg_ratios.mean()*100:.1f}%')
print(f'Visium HVG 비율: ~69.6%')

# Visium combined matrix 확인
import h5py
# Visium 첫 spot HVG 비율 확인
vis_expr = np.load('/project_antwerp/hbae/data/0317_HVG_NEW/combined_expression_matrix.npy')
print(f'\nVisium expm1 sum: {np.expm1(vis_expr[0]).sum():.0f}')


Open-ST 첫 cell: total counts=523, HVG counts=107
HVG 비율: 20.5%
전체 cell 평균 HVG 비율: 16.4%
Visium HVG 비율: ~69.6%

Visium expm1 sum: 6962


In [ ]:

import numpy as np

# Visium train_exprs 분석
train = np.load('/project_antwerp/hbae/Loki_output/0317_epoch10_finetune_embedding_new/fold_01/train_exprs.npy')
print('Visium train_exprs:')
print(f'  shape: {train.shape}')
print(f'  expm1 sum mean: {np.expm1(train).sum(axis=1).mean():.0f}')
print(f'  nonzero ratio: {(train>0).mean():.3f}')

# save_embeddings.py에서 어떻게 저장했는지
# gt_expr[spot_idx, hvg_indices] 로 저장
# gt_expr = combined_expression_matrix.npy (전체 spot x 전체 HVG)
# 즉 combined_expression_matrix 자체가 어떻게 만들어졌는지가 핵심

# combined_expression_matrix 분석
combined = np.load('/project_antwerp/hbae/data/0317_HVG_NEW/combined_expression_matrix.npy')
print(f'\ncombined_expression_matrix:')
print(f'  shape: {combined.shape}  (spots x all_shared_genes)')
print(f'  expm1 sum mean (첫 100): {np.expm1(combined[:100]).sum(axis=1).mean():.0f}')

# gene_list 확인
genes = open('/project_antwerp/hbae/data/0317_HVG_NEW/all_shared_genes.txt').read().strip().split('\n')
print(f'  all_shared_genes count: {len(genes)}')


In [1]:

import numpy as np
import scanpy as sc

adata = sc.read_h5ad('/project_antwerp/hbae/data/Open_ST/GSM7990099_primary_HNSCC.h5ad')

# 17368 shared genes 중 Open-ST에 있는 gene 수 확인
shared_genes = open('/project_antwerp/hbae/data/0317_HVG_NEW/all_shared_genes.txt').read().strip().split('\n')
openst_genes = set(adata.var_names)

overlap = [g for g in shared_genes if g in openst_genes]
print(f'17368 shared genes 중 Open-ST에 있는 gene: {len(overlap)}')
print(f'비율: {len(overlap)/len(shared_genes)*100:.1f}%')

# HVG 2000 중 shared genes에 있는 것
hvg_genes = open('/project_antwerp/hbae/data/0317_hvg_2000_list.txt').read().strip().split('\n')
hvg_in_shared = [g for g in hvg_genes if g in set(shared_genes)]
hvg_in_openst = [g for g in hvg_genes if g in openst_genes]
hvg_in_both   = [g for g in hvg_genes if g in set(shared_genes) and g in openst_genes]
print(f'\nHVG 2000:')
print(f'  shared genes에 있는 것: {len(hvg_in_shared)}')
print(f'  Open-ST에 있는 것: {len(hvg_in_openst)}')
print(f'  둘 다 있는 것: {len(hvg_in_both)}')

# Open-ST raw에서 shared genes 기준 normalize하면 expm1 sum이 얼마?
import scipy.sparse as sp
raw = adata.layers['raw']
if sp.issparse(raw): raw = raw.toarray()

gene_to_idx = {g: i for i, g in enumerate(adata.var_names)}
shared_in_openst_idx = [gene_to_idx[g] for g in shared_genes if g in gene_to_idx]
print(f'\nshared genes 중 Open-ST에 있는 index 수: {len(shared_in_openst_idx)}')

# shared genes 기준 normalize
spot = raw[0]
shared_sum = spot[shared_in_openst_idx].sum()
total_sum  = spot.sum()
print(f'첫 cell: total={total_sum:.0f}, shared_genes_sum={shared_sum:.0f}')
print(f'shared genes 비율: {shared_sum/total_sum*100:.1f}%')

# shared genes 기준 normalize 후 HVG subset expm1 sum
hvg_in_shared_openst = [g for g in hvg_genes if g in gene_to_idx and g in set(shared_genes)]
hvg_shared_idx = [gene_to_idx[g] for g in hvg_in_shared_openst]

norm_shared = spot[shared_in_openst_idx].sum()
if norm_shared > 0:
    normalized = spot / total_sum * 1e4  # 전체 기준
    hvg_expm1_sum = np.expm1(np.log1p(normalized[hvg_shared_idx])).sum()
    print(f'전체 기준 normalize 후 HVG expm1 sum: {hvg_expm1_sum:.0f}')


17368 shared genes 중 Open-ST에 있는 gene: 15010
비율: 86.4%

HVG 2000:
  shared genes에 있는 것: 2000
  Open-ST에 있는 것: 1946
  둘 다 있는 것: 1946

shared genes 중 Open-ST에 있는 index 수: 15010
첫 cell: total=523, shared_genes_sum=373
shared genes 비율: 71.3%
전체 기준 normalize 후 HVG expm1 sum: 2046


In [2]:

import numpy as np, h5py

with h5py.File('/project_antwerp/hbae/data/Open_ST/openst_patches_agg_mc10.h5', 'r') as f:
    expr = f['expression'][:100]

sums = np.expm1(expr).sum(axis=1)
print(f'현재 h5 expm1 sum: mean={sums.mean():.0f}, std={sums.std():.0f}')
print(f'목표: ~2046 (전체 gene 기준 normalize)')


현재 h5 expm1 sum: mean=2023, std=203
목표: ~2046 (전체 gene 기준 normalize)


In [1]:

import numpy as np
import scanpy as sc
import h5py

# Open-ST GT expression 로드
with h5py.File('/project_antwerp/hbae/data/Open_ST/openst_patches_agg_mc10.h5', 'r') as f:
    expr = f['expression'][:]
    gene_names_bytes = f.attrs['gene_names']
    genes = [g.decode() if isinstance(g, bytes) else g for g in gene_names_bytes]

print(f'Expression shape: {expr.shape}')

# AnnData 생성
adata = sc.AnnData(X=expr)
adata.var_names = genes

# HVG 선택 (Seurat flavor, log1p-normalized이므로)
sc.pp.highly_variable_genes(adata, n_top_genes=300, flavor='seurat')
hvg_300 = adata.var_names[adata.var['highly_variable']].tolist()
print(f'HVG 300 selected: {len(hvg_300)}')
print(f'Top-10: {hvg_300[:10]}')

# 현재 HEG top-300과 overlap 확인
mean_expr = expr.mean(axis=0)
heg_idx = np.argsort(mean_expr)[::-1][:300]
heg_300 = set([genes[i] for i in heg_idx])
hvg_set = set(hvg_300)
print(f'\nHEG top-300 vs HVG top-300 overlap: {len(heg_300 & hvg_set)}/300')
print(f'HVG에만 있는 gene: {len(hvg_set - heg_300)}')
print(f'HEG에만 있는 gene: {len(heg_300 - hvg_set)}')


Expression shape: (35041, 1946)
HVG 300 selected: 300
Top-10: ['A2M', 'A2ML1', 'ABLIM3', 'ACTA2', 'ACTG2', 'ACVRL1', 'ADGRF5', 'ADIRF', 'ADM', 'ALDH1A3']

HEG top-300 vs HVG top-300 overlap: 59/300
HVG에만 있는 gene: 241
HEG에만 있는 gene: 241


In [4]:
# 대신 지금 있는 결과로 top_pcc 근사치를 구할 수 있어
# fold_01 genewise_pcc.csv가 이미 있으면 그걸로 top-300 선택 가능


import pandas as pd
import numpy as np

# 이미 계산된 gene-wise PCC에서 top-300 선택
df = pd.read_csv('/project_antwerp/hbae/Loki_output/openst_validation_agg_v2/fold_01/openst_genewise_pcc.csv')
print('Already computed gene-wise PCC:')
print(f'  Total genes: {len(df)}')
print(f"  Top-10: {df.nlargest(10, 'pcc')['gene'].tolist()}")
print(f"  Top-300 mean PCC: {df.nlargest(300, 'pcc')['pcc'].mean():.4f}")
print(f"  Top-300 PCC range: {df.nlargest(300, 'pcc')['pcc'].min():.4f} ~ {df.nlargest(300, 'pcc')['pcc'].max():.4f}")


Already computed gene-wise PCC:
  Total genes: 300
  Top-10: ['LCE3E', 'CNFN', 'CDSN', 'CRCT1', 'DHRS9', 'PI3', 'IVL', 'RNASE7', 'IL1RN', 'SLPI']
  Top-300 mean PCC: 0.1157
  Top-300 PCC range: -0.1650 ~ 0.5503


In [6]:
# embedding 크기로 버전 확인

import numpy as np
import os

paths = [
    '/project_antwerp/hbae/Loki_output/openst_validation/fold_01/openst_img_embs.npy',
    '/project_antwerp/hbae/Loki_output/openst_validation_level3/fold_01/openst_img_embs.npy',
    '/project_antwerp/hbae/Loki_output/openst_validation_agg_v2/fold_01/openst_img_embs.npy',
]
for p in paths:
    if os.path.exists(p):
        emb = np.load(p)
        print(f"{p.split('/')[-3]}: {emb.shape}")
    else:
        print(f"{p.split('/')[-3]}: 없음")


openst_validation: (36040, 768)
openst_validation_level3: (36034, 768)
openst_validation_agg_v2: (35041, 768)


In [ ]:
python3 << 'EOF'
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

BASE = "/project_antwerp/hbae/Loki_output"

# ── 데이터 로드 ────────────────────────────────────────────────────────────
methods = {
    "HEG top-300\n(Loki 논문)": {
        "path": f"{BASE}/openst_validation_agg_v2/fold_{{:02d}}/openst_genewise_pcc.npy",
        "color": "#4C72B0"
    },
    "Scanpy HVG\ntop-300": {
        "path": f"{BASE}/openst_validation_hvg300/fold_{{:02d}}/openst_genewise_pcc.npy",
        "color": "#DD8452"
    },
    "Top-PCC\ntop-300": {
        "path": f"{BASE}/openst_validation_TopPCC300/fold_{{:02d}}/openst_genewise_pcc_toppcc.npy",
        "color": "#55A868"
    },
}

data = {}   # method → list of arrays (fold별)
for method, info in methods.items():
    fold_data = []
    for fold in range(1, 11):
        path = info["path"].format(fold)
        try:
            arr = np.load(path)
            fold_data.append(arr)
        except:
            print(f"  Missing: {path}")
    data[method] = fold_data
    print(f"{method.replace(chr(10),' ')}: {len(fold_data)} folds loaded, "
          f"mean PCC = {np.mean([a.mean() for a in fold_data]):.4f}")

# ── 그림 ──────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 7), sharey=False)
fig.suptitle("Open-ST HNSCC External Validation\nGene-wise PCC Distribution by Fold",
             fontsize=14, fontweight='bold', y=1.01)

for ax, (method, info) in zip(axes, methods.items()):
    fold_arrays = data[method]
    color = info["color"]

    # violin plot
    parts = ax.violinplot(
        fold_arrays,
        positions=range(1, len(fold_arrays) + 1),
        showmeans=True, showmedians=True, showextrema=True
    )

    # 색상 적용
    for pc in parts['bodies']:
        pc.set_facecolor(color)
        pc.set_alpha(0.7)
    for part in ['cmeans', 'cmedians', 'cbars', 'cmins', 'cmaxes']:
        if part in parts:
            parts[part].set_color('black')
            parts[part].set_linewidth(1.2)
    parts['cmeans'].set_color('red')
    parts['cmeans'].set_linewidth(2)

    # 각 fold mean 값 표시
    means = [a.mean() for a in fold_arrays]
    ax.scatter(range(1, len(fold_arrays) + 1), means,
               color='red', zorder=5, s=30, label='mean')

    # overall mean line
    overall_mean = np.mean(means)
    ax.axhline(overall_mean, color=color, linestyle='--', linewidth=1.5,
               label=f'overall mean={overall_mean:.3f}')
    ax.axhline(0, color='gray', linestyle='-', linewidth=0.8, alpha=0.5)

    ax.set_xlabel('Fold', fontsize=11)
    ax.set_ylabel('Gene-wise PCC', fontsize=11)
    ax.set_title(method, fontsize=12, fontweight='bold', color=info["color"])
    ax.set_xticks(range(1, len(fold_arrays) + 1))
    ax.set_xticklabels([f'F{i:02d}' for i in range(1, len(fold_arrays) + 1)],
                       fontsize=8, rotation=45)
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
out = "/project_antwerp/hbae/Loki_output/openst_validation_violin_genewise.png"
plt.savefig(out, dpi=150, bbox_inches='tight')
print(f"\nSaved: {out}")

# ── 통합 violin (3 방식 나란히) ────────────────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(14, 7))

all_data_flat = []
labels = []
colors_flat = []
positions = []

pos = 1
method_centers = {}
for method, info in methods.items():
    fold_arrays = data[method]
    color = info["color"]
    start_pos = pos
    for arr in fold_arrays:
        all_data_flat.append(arr)
        colors_flat.append(color)
        positions.append(pos)
        pos += 1
    method_centers[method] = (start_pos + pos - 1) / 2
    pos += 1.5  # 방식 간 간격

parts2 = ax2.violinplot(all_data_flat, positions=positions,
                        showmeans=True, showmedians=False, showextrema=False,
                        widths=0.8)
for i, pc in enumerate(parts2['bodies']):
    pc.set_facecolor(colors_flat[i])
    pc.set_alpha(0.65)
parts2['cmeans'].set_color('red')
parts2['cmeans'].set_linewidth(1.5)

# 방식 구분선 및 레이블
for method, info in methods.items():
    fold_arrays = data[method]
    overall_mean = np.mean([a.mean() for a in fold_arrays])
    cx = method_centers[method]
    ax2.text(cx, ax2.get_ylim()[0] if ax2.get_ylim()[0] != 0 else -0.25,
             method.replace('\n', ' '), ha='center', va='top', fontsize=9,
             color=info["color"], fontweight='bold')

ax2.axhline(0, color='gray', linestyle='-', linewidth=0.8, alpha=0.5)
ax2.set_ylabel('Gene-wise PCC', fontsize=12)
ax2.set_title("Open-ST External Validation — Gene-wise PCC\n(All Methods, All Folds)",
              fontsize=13, fontweight='bold')
ax2.set_xticks(positions)
ax2.set_xticklabels([f'F{((i)%10)+1:02d}' for i in range(len(positions))],
                    fontsize=7, rotation=45)

legend_patches = [mpatches.Patch(color=info["color"],
                  label=m.replace('\n', ' ') + f' (mean={np.mean([a.mean() for a in data[m]]):.3f})')
                  for m, info in methods.items()]
ax2.legend(handles=legend_patches, fontsize=9, loc='upper right')
ax2.grid(axis='y', alpha=0.3)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

plt.tight_layout()
out2 = "/project_antwerp/hbae/Loki_output/openst_validation_violin_combined.png"
plt.savefig(out2, dpi=150, bbox_inches='tight')
print(f"Saved: {out2}")
EOF